# Banking Assignment — Exception Handling (Solved)

100 questions covering accounts, deposits, withdrawals, transfers, cards, loans,
KYC, fraud monitoring, reconciliation, and audit trails.

**Notes on approach**
- Every function raises the *narrowest* exception that actually applies (`ValueError`,
  `KeyError`, `ZeroDivisionError`, etc.) for programming/input errors, and a *custom*
  exception for business-rule failures (insufficient funds, invalid KYC, etc.).
- Where a lower-level exception is re-raised as a higher-level one, `raise NewErr(...) from exc`
  is used so the original cause is preserved in the traceback.
- No real customer data, PINs, credentials, or full card numbers are stored — only masked
  or synthetic reference values (e.g. `ACC-1001`) are used.
- Each solution ends with a short demo so you can see normal + edge-case behaviour.


## Shared exception hierarchy
Defined once here and reused by later cells.

In [1]:
class BankingError(Exception):
    """Base class for all business-rule (not programming) failures in this notebook."""


class InsufficientFundsError(BankingError):
    """Raised when a withdrawal/transfer would take a balance below its allowed minimum."""


class InvalidTransactionError(BankingError):
    """Raised when a transaction's shape/data is invalid, chained from the low-level cause."""


class AccountNotFoundError(BankingError):
    """Raised when an account reference cannot be located."""


class KYCValidationError(BankingError):
    """Raised when one or more KYC fields fail validation. May hold several sub-errors."""

    def __init__(self, account_ref, problems):
        self.account_ref = account_ref
        self.problems = problems
        super().__init__(f"[{account_ref}] KYC failed: {'; '.join(problems)}")


class TransientGatewayError(BankingError):
    """Raised for a payment-gateway failure that is safe to retry (timeouts, 5xx, etc.)."""


print("Shared exception hierarchy ready:", [c.__name__ for c in
      (BankingError, InsufficientFundsError, InvalidTransactionError,
       AccountNotFoundError, KYCValidationError, TransientGatewayError)])


Shared exception hierarchy ready: ['BankingError', 'InsufficientFundsError', 'InvalidTransactionError', 'AccountNotFoundError', 'KYCValidationError', 'TransientGatewayError']


## Easy — Fundamentals and direct applications (Q1–Q30)

### Q1. Parse a deposit amount for banking case 1; use sample reference ACC-1001 and explain the result.

In [ ]:
def parse_deposit_amount(raw_amount, account_ref="ACC-1001"):
    """Parse a user-supplied deposit amount string into a float."""
    try:
        amount = float(raw_amount)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"[{account_ref}] '{raw_amount}' is not a valid deposit amount") from exc
    if amount <= 0:
        raise ValueError(f"[{account_ref}] Deposit amount must be positive, got {amount}")
    return amount


for test_value in ["250.50", "", "abc", -10, None]:
    try:
        result = parse_deposit_amount(test_value)
        print(f"OK      -> parsed deposit of {result} for ACC-1001")
    except ValueError as e:
        print(f"REJECTED-> {e}")




OK      -> parsed deposit of 250.5 for ACC-1001
REJECTED-> [ACC-1001] '' is not a valid deposit amount
REJECTED-> [ACC-1001] 'abc' is not a valid deposit amount
REJECTED-> [ACC-1001] Deposit amount must be positive, got -10.0
REJECTED-> [ACC-1001] 'None' is not a valid deposit amount


### Q2. Handle a missing account key for banking case 1; use sample reference ACC-1002 and explain the result.

In [ ]:
accounts = {
    "ACC-1002": {"balance": 1500.0},
}

def get_balance(account_ref):
    try:
        return accounts[account_ref]["balance"]
    except KeyError as exc:
        raise AccountNotFoundError(f"Account '{account_ref}' does not exist") from exc


for ref in ["ACC-1002", "ACC-9999"]:
    try:
        print(f"{ref} balance = {get_balance(ref)}")
    except AccountNotFoundError as e:
        print(f"REJECTED-> {e}")



ACC-1002 balance = 1500.0
REJECTED-> Account 'ACC-9999' does not exist


### Q3. Prevent division by zero in average balance for banking case 1; use sample reference ACC-1003 and explain the result.

In [ ]:
def average_balance(daily_balances, account_ref="ACC-1003"):
    try:
        return sum(daily_balances) / len(daily_balances)
    except ZeroDivisionError as exc:
        raise ValueError(f"[{account_ref}] Cannot average an empty balance history") from exc


for history in [[1000, 1200, 900], []]:
    try:
        print(f"Average for ACC-1003: {average_balance(history)}")
    except ValueError as e:
        print(f"REJECTED-> {e}")




Average for ACC-1003: 1033.3333333333333
REJECTED-> [ACC-1003] Cannot average an empty balance history


### Q4. Open a transaction file safely for banking case 1; use sample reference ACC-1004 and explain the result.

In [ ]:
def read_transaction_log(path, account_ref="ACC-1004"):
    try:
        with open(path, "r") as fh:
            return fh.read()
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"[{account_ref}] Transaction log not found at '{path}'") from exc
    except PermissionError as exc:
        raise PermissionError(f"[{account_ref}] No permission to read '{path}'") from exc


for path in ["/tmp/ACC-1004_missing.log"]:
    try:
        read_transaction_log(path)
    except (FileNotFoundError, PermissionError) as e:
        print(f"REJECTED-> {e}")




REJECTED-> [ACC-1004] Transaction log not found at '/tmp/ACC-1004_missing.log'


### Q5. Reject a negative withdrawal for banking case 1; use sample reference ACC-1005 and explain the result.

In [ ]:
def validate_withdrawal(amount, account_ref="ACC-1005"):
    if not isinstance(amount, (int, float)):
        raise TypeError(f"[{account_ref}] Withdrawal amount must be numeric, got {type(amount).__name__}")
    if amount < 0:
        raise ValueError(f"[{account_ref}] Withdrawal amount cannot be negative: {amount}")
    return amount


for value in [500, -50, "100"]:
    try:
        print(f"Withdrawal of {validate_withdrawal(value)} accepted for ACC-1005")
    except (TypeError, ValueError) as e:
        print(f"REJECTED-> {e}")




Withdrawal of 500 accepted for ACC-1005
REJECTED-> [ACC-1005] Withdrawal amount cannot be negative: -50
REJECTED-> [ACC-1005] Withdrawal amount must be numeric, got str


### Q6. Catch invalid tenure input for banking case 1; use sample reference ACC-1006 and explain the result.

In [ ]:
def parse_loan_tenure_months(raw_tenure, account_ref="ACC-1006"):
    try:
        tenure = int(raw_tenure)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"[{account_ref}] '{raw_tenure}' is not a valid tenure in months") from exc
    if not (1 <= tenure <= 360):
        raise ValueError(f"[{account_ref}] Tenure must be between 1 and 360 months, got {tenure}")
    return tenure


for value in ["36", "0", "abc", "480"]:
    try:
        print(f"Tenure {parse_loan_tenure_months(value)} months accepted for ACC-1006")
    except ValueError as e:
        print(f"REJECTED-> {e}")




Tenure 36 months accepted for ACC-1006
REJECTED-> [ACC-1006] Tenure must be between 1 and 360 months, got 0
REJECTED-> [ACC-1006] 'abc' is not a valid tenure in months
REJECTED-> [ACC-1006] Tenure must be between 1 and 360 months, got 480


### Q7. Use else after pin-format validation for banking case 1; use sample reference ACC-1007 and explain the result.

In [ ]:
def validate_pin_format(pin, account_ref="ACC-1007"):
    try:
        pin_int = int(pin)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"[{account_ref}] PIN must be numeric") from exc
    else:
        # Only runs if the try block succeeded with no exception.
        if len(str(pin)) != 4:
            raise ValueError(f"[{account_ref}] PIN must be exactly 4 digits")
        print(f"[{account_ref}] PIN format accepted (masked): **{str(pin)[-2:]}")
        return True


for value in ["1234", "12", "abcd"]:
    try:
        validate_pin_format(value)
    except ValueError as e:
        print(f"REJECTED-> {e}")




[ACC-1007] PIN format accepted (masked): **34
REJECTED-> [ACC-1007] PIN must be exactly 4 digits
REJECTED-> [ACC-1007] PIN must be numeric


### Q8. Use finally to close a session for banking case 1; use sample reference ACC-1008 and explain the result.

In [ ]:
class BankSession:
    def __init__(self, account_ref):
        self.account_ref = account_ref
        self.open = False

    def open_session(self):
        self.open = True
        print(f"[{self.account_ref}] session opened")

    def close_session(self):
        self.open = False
        print(f"[{self.account_ref}] session closed")


def run_in_session(account_ref, action):
    session = BankSession(account_ref)
    session.open_session()
    try:
        return action()
    finally:
        # Runs whether action() succeeds, raises, or returns early -- guarantees cleanup.
        session.close_session()


try:
    run_in_session("ACC-1008", lambda: (_ for _ in ()).throw(ValueError("simulated failure")))
except ValueError as e:
    print(f"REJECTED-> {e}")




[ACC-1008] session opened
[ACC-1008] session closed
REJECTED-> simulated failure


### Q9. Catch multiple numeric conversion errors for banking case 1; use sample reference ACC-1009 and explain the result.

In [ ]:
def parse_interest_rate(raw_rate, account_ref="ACC-1009"):
    try:
        rate = float(raw_rate)
        if rate != rate:  # NaN check
            raise ValueError("rate is NaN")
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError(f"[{account_ref}] '{raw_rate}' is not a usable interest rate") from exc
    return rate


for value in ["7.5", "nan", None, "1e400"]:
    try:
        print(f"Rate {parse_interest_rate(value)}% accepted for ACC-1009")
    except ValueError as e:
        print(f"REJECTED-> {e}")




Rate 7.5% accepted for ACC-1009
REJECTED-> [ACC-1009] 'nan' is not a usable interest rate
REJECTED-> [ACC-1009] 'None' is not a usable interest rate
Rate inf% accepted for ACC-1009


### Q10. Raise a clear valueerror for banking case 1; use sample reference ACC-1010 and explain the result.

In [ ]:
def set_account_currency(currency_code, account_ref="ACC-1010"):
    allowed = {"INR", "USD", "EUR", "GBP"}
    if not isinstance(currency_code, str) or currency_code.upper() not in allowed:
        raise ValueError(
            f"[{account_ref}] '{currency_code}' is not a supported currency; "
            f"choose one of {sorted(allowed)}"
        )
    return currency_code.upper()


for value in ["inr", "XYZ", 123]:
    try:
        print(f"Currency set to {set_account_currency(value)} for ACC-1010")
    except ValueError as e:
        print(f"REJECTED-> {e}")




Currency set to INR for ACC-1010
REJECTED-> [ACC-1010] 'XYZ' is not a supported currency; choose one of ['EUR', 'GBP', 'INR', 'USD']
REJECTED-> [ACC-1010] '123' is not a supported currency; choose one of ['EUR', 'GBP', 'INR', 'USD']


### Q11. Parse a deposit amount for banking case 2; use sample reference ACC-1011 and explain the result.

In [ ]:
def parse_deposit_amount(raw_amount, account_ref="ACC-1011"):
    """Parse a user-supplied deposit amount string into a float."""
    try:
        amount = float(raw_amount)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"[{account_ref}] '{raw_amount}' is not a valid deposit amount") from exc
    if amount <= 0:
        raise ValueError(f"[{account_ref}] Deposit amount must be positive, got {amount}")
    return amount


for test_value in ["250.50", "", "abc", -10, None]:
    try:
        result = parse_deposit_amount(test_value)
        print(f"OK      -> parsed deposit of {result} for ACC-1011")
    except ValueError as e:
        print(f"REJECTED-> {e}")




OK      -> parsed deposit of 250.5 for ACC-1011
REJECTED-> [ACC-1011] '' is not a valid deposit amount
REJECTED-> [ACC-1011] 'abc' is not a valid deposit amount
REJECTED-> [ACC-1011] Deposit amount must be positive, got -10.0
REJECTED-> [ACC-1011] 'None' is not a valid deposit amount


### Q12. Handle a missing account key for banking case 2; use sample reference ACC-1012 and explain the result.

In [ ]:
accounts = {
    "ACC-1012": {"balance": 1500.0},
}

def get_balance(account_ref):
    try:
        return accounts[account_ref]["balance"]
    except KeyError as exc:
        raise AccountNotFoundError(f"Account '{account_ref}' does not exist") from exc


for ref in ["ACC-1012", "ACC-9999"]:
    try:
        print(f"{ref} balance = {get_balance(ref)}")
    except AccountNotFoundError as e:
        print(f"REJECTED-> {e}")



ACC-1012 balance = 1500.0
REJECTED-> Account 'ACC-9999' does not exist


### Q13. Prevent division by zero in average balance for banking case 2; use sample reference ACC-1013 and explain the result.

In [ ]:
def average_balance(daily_balances, account_ref="ACC-1013"):
    try:
        return sum(daily_balances) / len(daily_balances)
    except ZeroDivisionError as exc:
        raise ValueError(f"[{account_ref}] Cannot average an empty balance history") from exc


for history in [[1000, 1200, 900], []]:
    try:
        print(f"Average for ACC-1013: {average_balance(history)}")
    except ValueError as e:
        print(f"REJECTED-> {e}")

.


Average for ACC-1013: 1033.3333333333333
REJECTED-> [ACC-1013] Cannot average an empty balance history


### Q14. Open a transaction file safely for banking case 2; use sample reference ACC-1014 and explain the result.

In [ ]:
def read_transaction_log(path, account_ref="ACC-1014"):
    try:
        with open(path, "r") as fh:
            return fh.read()
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"[{account_ref}] Transaction log not found at '{path}'") from exc
    except PermissionError as exc:
        raise PermissionError(f"[{account_ref}] No permission to read '{path}'") from exc


for path in ["/tmp/ACC-1014_missing.log"]:
    try:
        read_transaction_log(path)
    except (FileNotFoundError, PermissionError) as e:
        print(f"REJECTED-> {e}")



REJECTED-> [ACC-1014] Transaction log not found at '/tmp/ACC-1014_missing.log'


### Q15. Reject a negative withdrawal for banking case 2; use sample reference ACC-1015 and explain the result.

In [ ]:
def validate_withdrawal(amount, account_ref="ACC-1015"):
    if not isinstance(amount, (int, float)):
        raise TypeError(f"[{account_ref}] Withdrawal amount must be numeric, got {type(amount).__name__}")
    if amount < 0:
        raise ValueError(f"[{account_ref}] Withdrawal amount cannot be negative: {amount}")
    return amount


for value in [500, -50, "100"]:
    try:
        print(f"Withdrawal of {validate_withdrawal(value)} accepted for ACC-1015")
    except (TypeError, ValueError) as e:
        print(f"REJECTED-> {e}")




Withdrawal of 500 accepted for ACC-1015
REJECTED-> [ACC-1015] Withdrawal amount cannot be negative: -50
REJECTED-> [ACC-1015] Withdrawal amount must be numeric, got str


### Q16. Catch invalid tenure input for banking case 2; use sample reference ACC-1016 and explain the result.

In [ ]:
def parse_loan_tenure_months(raw_tenure, account_ref="ACC-1016"):
    try:
        tenure = int(raw_tenure)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"[{account_ref}] '{raw_tenure}' is not a valid tenure in months") from exc
    if not (1 <= tenure <= 360):
        raise ValueError(f"[{account_ref}] Tenure must be between 1 and 360 months, got {tenure}")
    return tenure


for value in ["36", "0", "abc", "480"]:
    try:
        print(f"Tenure {parse_loan_tenure_months(value)} months accepted for ACC-1016")
    except ValueError as e:
        print(f"REJECTED-> {e}")




Tenure 36 months accepted for ACC-1016
REJECTED-> [ACC-1016] Tenure must be between 1 and 360 months, got 0
REJECTED-> [ACC-1016] 'abc' is not a valid tenure in months
REJECTED-> [ACC-1016] Tenure must be between 1 and 360 months, got 480


### Q17. Use else after pin-format validation for banking case 2; use sample reference ACC-1017 and explain the result.

In [ ]:
def validate_pin_format(pin, account_ref="ACC-1017"):
    try:
        pin_int = int(pin)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"[{account_ref}] PIN must be numeric") from exc
    else:
        # Only runs if the try block succeeded with no exception.
        if len(str(pin)) != 4:
            raise ValueError(f"[{account_ref}] PIN must be exactly 4 digits")
        print(f"[{account_ref}] PIN format accepted (masked): **{str(pin)[-2:]}")
        return True


for value in ["1234", "12", "abcd"]:
    try:
        validate_pin_format(value)
    except ValueError as e:
        print(f"REJECTED-> {e}")



[ACC-1017] PIN format accepted (masked): **34
REJECTED-> [ACC-1017] PIN must be exactly 4 digits
REJECTED-> [ACC-1017] PIN must be numeric


### Q18. Use finally to close a session for banking case 2; use sample reference ACC-1018 and explain the result.

In [ ]:
class BankSession:
    def __init__(self, account_ref):
        self.account_ref = account_ref
        self.open = False

    def open_session(self):
        self.open = True
        print(f"[{self.account_ref}] session opened")

    def close_session(self):
        self.open = False
        print(f"[{self.account_ref}] session closed")


def run_in_session(account_ref, action):
    session = BankSession(account_ref)
    session.open_session()
    try:
        return action()
    finally:
        # Runs whether action() succeeds, raises, or returns early -- guarantees cleanup.
        session.close_session()


try:
    run_in_session("ACC-1018", lambda: (_ for _ in ()).throw(ValueError("simulated failure")))
except ValueError as e:
    print(f"REJECTED-> {e}")




[ACC-1018] session opened
[ACC-1018] session closed
REJECTED-> simulated failure


### Q19. Catch multiple numeric conversion errors for banking case 2; use sample reference ACC-1019 and explain the result.

In [ ]:
def parse_interest_rate(raw_rate, account_ref="ACC-1019"):
    try:
        rate = float(raw_rate)
        if rate != rate:  # NaN check
            raise ValueError("rate is NaN")
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError(f"[{account_ref}] '{raw_rate}' is not a usable interest rate") from exc
    return rate


for value in ["7.5", "nan", None, "1e400"]:
    try:
        print(f"Rate {parse_interest_rate(value)}% accepted for ACC-1019")
    except ValueError as e:
        print(f"REJECTED-> {e}")




Rate 7.5% accepted for ACC-1019
REJECTED-> [ACC-1019] 'nan' is not a usable interest rate
REJECTED-> [ACC-1019] 'None' is not a usable interest rate
Rate inf% accepted for ACC-1019


### Q20. Raise a clear valueerror for banking case 2; use sample reference ACC-1020 and explain the result.

In [ ]:
def set_account_currency(currency_code, account_ref="ACC-1020"):
    allowed = {"INR", "USD", "EUR", "GBP"}
    if not isinstance(currency_code, str) or currency_code.upper() not in allowed:
        raise ValueError(
            f"[{account_ref}] '{currency_code}' is not a supported currency; "
            f"choose one of {sorted(allowed)}"
        )
    return currency_code.upper()


for value in ["inr", "XYZ", 123]:
    try:
        print(f"Currency set to {set_account_currency(value)} for ACC-1020")
    except ValueError as e:
        print(f"REJECTED-> {e}")




Currency set to INR for ACC-1020
REJECTED-> [ACC-1020] 'XYZ' is not a supported currency; choose one of ['EUR', 'GBP', 'INR', 'USD']
REJECTED-> [ACC-1020] '123' is not a supported currency; choose one of ['EUR', 'GBP', 'INR', 'USD']


### Q21. Parse a deposit amount for banking case 3; use sample reference ACC-1021 and explain the result.

In [ ]:
def parse_deposit_amount(raw_amount, account_ref="ACC-1021"):
    """Parse a user-supplied deposit amount string into a float."""
    try:
        amount = float(raw_amount)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"[{account_ref}] '{raw_amount}' is not a valid deposit amount") from exc
    if amount <= 0:
        raise ValueError(f"[{account_ref}] Deposit amount must be positive, got {amount}")
    return amount


for test_value in ["250.50", "", "abc", -10, None]:
    try:
        result = parse_deposit_amount(test_value)
        print(f"OK      -> parsed deposit of {result} for ACC-1021")
    except ValueError as e:
        print(f"REJECTED-> {e}")




OK      -> parsed deposit of 250.5 for ACC-1021
REJECTED-> [ACC-1021] '' is not a valid deposit amount
REJECTED-> [ACC-1021] 'abc' is not a valid deposit amount
REJECTED-> [ACC-1021] Deposit amount must be positive, got -10.0
REJECTED-> [ACC-1021] 'None' is not a valid deposit amount


### Q22. Handle a missing account key for banking case 3; use sample reference ACC-1022 and explain the result.

In [ ]:
accounts = {
    "ACC-1022": {"balance": 1500.0},
}

def get_balance(account_ref):
    try:
        return accounts[account_ref]["balance"]
    except KeyError as exc:
        raise AccountNotFoundError(f"Account '{account_ref}' does not exist") from exc


for ref in ["ACC-1022", "ACC-9999"]:
    try:
        print(f"{ref} balance = {get_balance(ref)}")
    except AccountNotFoundError as e:
        print(f"REJECTED-> {e}")




ACC-1022 balance = 1500.0
REJECTED-> Account 'ACC-9999' does not exist


### Q23. Prevent division by zero in average balance for banking case 3; use sample reference ACC-1023 and explain the result.

In [ ]:
def average_balance(daily_balances, account_ref="ACC-1023"):
    try:
        return sum(daily_balances) / len(daily_balances)
    except ZeroDivisionError as exc:
        raise ValueError(f"[{account_ref}] Cannot average an empty balance history") from exc


for history in [[1000, 1200, 900], []]:
    try:
        print(f"Average for ACC-1023: {average_balance(history)}")
    except ValueError as e:
        print(f"REJECTED-> {e}")



Average for ACC-1023: 1033.3333333333333
REJECTED-> [ACC-1023] Cannot average an empty balance history


### Q24. Open a transaction file safely for banking case 3; use sample reference ACC-1024 and explain the result.

In [ ]:
def read_transaction_log(path, account_ref="ACC-1024"):
    try:
        with open(path, "r") as fh:
            return fh.read()
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"[{account_ref}] Transaction log not found at '{path}'") from exc
    except PermissionError as exc:
        raise PermissionError(f"[{account_ref}] No permission to read '{path}'") from exc


for path in ["/tmp/ACC-1024_missing.log"]:
    try:
        read_transaction_log(path)
    except (FileNotFoundError, PermissionError) as e:
        print(f"REJECTED-> {e}")




REJECTED-> [ACC-1024] Transaction log not found at '/tmp/ACC-1024_missing.log'


### Q25. Reject a negative withdrawal for banking case 3; use sample reference ACC-1025 and explain the result.

In [ ]:
def validate_withdrawal(amount, account_ref="ACC-1025"):
    if not isinstance(amount, (int, float)):
        raise TypeError(f"[{account_ref}] Withdrawal amount must be numeric, got {type(amount).__name__}")
    if amount < 0:
        raise ValueError(f"[{account_ref}] Withdrawal amount cannot be negative: {amount}")
    return amount


for value in [500, -50, "100"]:
    try:
        print(f"Withdrawal of {validate_withdrawal(value)} accepted for ACC-1025")
    except (TypeError, ValueError) as e:
        print(f"REJECTED-> {e}")



Withdrawal of 500 accepted for ACC-1025
REJECTED-> [ACC-1025] Withdrawal amount cannot be negative: -50
REJECTED-> [ACC-1025] Withdrawal amount must be numeric, got str


### Q26. Catch invalid tenure input for banking case 3; use sample reference ACC-1026 and explain the result.

In [ ]:
def parse_loan_tenure_months(raw_tenure, account_ref="ACC-1026"):
    try:
        tenure = int(raw_tenure)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"[{account_ref}] '{raw_tenure}' is not a valid tenure in months") from exc
    if not (1 <= tenure <= 360):
        raise ValueError(f"[{account_ref}] Tenure must be between 1 and 360 months, got {tenure}")
    return tenure


for value in ["36", "0", "abc", "480"]:
    try:
        print(f"Tenure {parse_loan_tenure_months(value)} months accepted for ACC-1026")
    except ValueError as e:
        print(f"REJECTED-> {e}")


Tenure 36 months accepted for ACC-1026
REJECTED-> [ACC-1026] Tenure must be between 1 and 360 months, got 0
REJECTED-> [ACC-1026] 'abc' is not a valid tenure in months
REJECTED-> [ACC-1026] Tenure must be between 1 and 360 months, got 480


### Q27. Use else after pin-format validation for banking case 3; use sample reference ACC-1027 and explain the result.

In [ ]:
def validate_pin_format(pin, account_ref="ACC-1027"):
    try:
        pin_int = int(pin)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"[{account_ref}] PIN must be numeric") from exc
    else:
        # Only runs if the try block succeeded with no exception.
        if len(str(pin)) != 4:
            raise ValueError(f"[{account_ref}] PIN must be exactly 4 digits")
        print(f"[{account_ref}] PIN format accepted (masked): **{str(pin)[-2:]}")
        return True


for value in ["1234", "12", "abcd"]:
    try:
        validate_pin_format(value)
    except ValueError as e:
        print(f"REJECTED-> {e}")



[ACC-1027] PIN format accepted (masked): **34
REJECTED-> [ACC-1027] PIN must be exactly 4 digits
REJECTED-> [ACC-1027] PIN must be numeric


### Q28. Use finally to close a session for banking case 3; use sample reference ACC-1028 and explain the result.

In [ ]:
class BankSession:
    def __init__(self, account_ref):
        self.account_ref = account_ref
        self.open = False

    def open_session(self):
        self.open = True
        print(f"[{self.account_ref}] session opened")

    def close_session(self):
        self.open = False
        print(f"[{self.account_ref}] session closed")


def run_in_session(account_ref, action):
    session = BankSession(account_ref)
    session.open_session()
    try:
        return action()
    finally:
        # Runs whether action() succeeds, raises, or returns early -- guarantees cleanup.
        session.close_session()


try:
    run_in_session("ACC-1028", lambda: (_ for _ in ()).throw(ValueError("simulated failure")))
except ValueError as e:
    print(f"REJECTED-> {e}")




[ACC-1028] session opened
[ACC-1028] session closed
REJECTED-> simulated failure


### Q29. Catch multiple numeric conversion errors for banking case 3; use sample reference ACC-1029 and explain the result.

In [ ]:
def parse_interest_rate(raw_rate, account_ref="ACC-1029"):
    try:
        rate = float(raw_rate)
        if rate != rate:  # NaN check
            raise ValueError("rate is NaN")
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError(f"[{account_ref}] '{raw_rate}' is not a usable interest rate") from exc
    return rate


for value in ["7.5", "nan", None, "1e400"]:
    try:
        print(f"Rate {parse_interest_rate(value)}% accepted for ACC-1029")
    except ValueError as e:
        print(f"REJECTED-> {e}")



Rate 7.5% accepted for ACC-1029
REJECTED-> [ACC-1029] 'nan' is not a usable interest rate
REJECTED-> [ACC-1029] 'None' is not a usable interest rate
Rate inf% accepted for ACC-1029


### Q30. Raise a clear valueerror for banking case 3; use sample reference ACC-1030 and explain the result.

In [ ]:
def set_account_currency(currency_code, account_ref="ACC-1030"):
    allowed = {"INR", "USD", "EUR", "GBP"}
    if not isinstance(currency_code, str) or currency_code.upper() not in allowed:
        raise ValueError(
            f"[{account_ref}] '{currency_code}' is not a supported currency; "
            f"choose one of {sorted(allowed)}"
        )
    return currency_code.upper()


for value in ["inr", "XYZ", 123]:
    try:
        print(f"Currency set to {set_account_currency(value)} for ACC-1030")
    except ValueError as e:
        print(f"REJECTED-> {e}")




Currency set to INR for ACC-1030
REJECTED-> [ACC-1030] 'XYZ' is not a supported currency; choose one of ['EUR', 'GBP', 'INR', 'USD']
REJECTED-> [ACC-1030] '123' is not a supported currency; choose one of ['EUR', 'GBP', 'INR', 'USD']


## Medium — Reusable components and multi-step workflows (Q31–Q70)

### Q31. Create insufficientfundserror for banking case 1; use sample reference ACC-1031 and explain the result.

In [ ]:
def withdraw(balance, amount, account_ref="ACC-1031", min_balance=0):
    if amount > balance - min_balance:
        raise InsufficientFundsError(
            f"[{account_ref}] Cannot withdraw {amount}: balance {balance} "
            f"would fall below minimum {min_balance}"
        )
    return balance - amount


try:
    new_balance = withdraw(1000, 1500)
except InsufficientFundsError as e:
    print(f"REJECTED-> {e}")
else:
    print(f"New balance for ACC-1031: {new_balance}")




REJECTED-> [ACC-1031] Cannot withdraw 1500: balance 1000 would fall below minimum 0


### Q32. Chain invalidtransactionerror from valueerror for banking case 1; use sample reference ACC-1032 and explain the result.

In [ ]:
def build_transaction(raw_amount, account_ref="ACC-1032"):
    try:
        amount = float(raw_amount)
        if amount <= 0:
            raise ValueError("amount must be positive")
    except ValueError as exc:
        raise InvalidTransactionError(f"[{account_ref}] Could not build transaction") from exc
    return {"account_ref": account_ref, "amount": amount}


try:
    build_transaction("-5")
except InvalidTransactionError as e:
    print(f"REJECTED-> {e}")
    print(f"  caused by: {e.__cause__!r}")

.


REJECTED-> [ACC-1032] Could not build transaction
  caused by: ValueError('amount must be positive')


### Q33. Process transfers while isolating bad rows for banking case 1; use sample reference ACC-1033 and explain the result.

In [ ]:
def process_transfer_batch(rows, account_ref="ACC-1033"):
    successes, failures = [], []
    for i, row in enumerate(rows):
        try:
            amount = float(row["amount"])
            if amount <= 0:
                raise ValueError("non-positive transfer amount")
            successes.append({"row": i, "amount": amount})
        except (KeyError, ValueError) as exc:
            failures.append({"row": i, "error": str(exc)})
    return successes, failures


rows = [{"amount": "100"}, {"amount": "-5"}, {"note": "missing amount"}, {"amount": "250"}]
ok, bad = process_transfer_batch(rows)
print(f"ACC-1033: {len(ok)} succeeded, {len(bad)} failed")
for f in bad:
    print(f"  REJECTED row {f['row']}-> {f['error']}")




ACC-1033: 2 succeeded, 2 failed
  REJECTED row 1-> non-positive transfer amount
  REJECTED row 2-> 'amount'


### Q34. Retry a transient payment gateway for banking case 1; use sample reference ACC-1034 and explain the result.

In [ ]:
import random

def call_payment_gateway(account_ref="ACC-1034", _attempts_before_success=2, _state={"n": 0}):
    _state["n"] += 1
    if _state["n"] <= _attempts_before_success:
        raise TransientGatewayError(f"[{account_ref}] gateway timeout (attempt {_state['n']})")
    return f"payment-confirmed-{account_ref}"


def call_with_retry(fn, max_attempts=3):
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except TransientGatewayError as exc:
            last_exc = exc
            print(f"  retrying after: {exc}")
    raise last_exc


print(call_with_retry(call_payment_gateway))



  retrying after: [ACC-1034] gateway timeout (attempt 1)
  retrying after: [ACC-1034] gateway timeout (attempt 2)
payment-confirmed-ACC-1034


### Q35. Log failed standing instructions for banking case 1; use sample reference ACC-1035 and explain the result.

In [3]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("ACC-1035")


# Define the missing custom exception
class InvalidTransactionError(Exception):
    pass


def execute_standing_instruction(instruction, account_ref="ACC-1035"):
    try:
        if instruction["amount"] <= 0:
            raise InvalidTransactionError(
                f"[{account_ref}] standing instruction amount must be positive"
            )
        return f"executed {instruction['amount']} for {account_ref}"
    except InvalidTransactionError:
        logger.error(
            "Standing instruction failed for %s: %s",
            account_ref,
            instruction,
            exc_info=True,
        )
        raise


try:
    execute_standing_instruction({"amount": -100})
except InvalidTransactionError as e:
    print(f"REJECTED (already logged)-> {e}")




ERROR: Standing instruction failed for ACC-1035: {'amount': -100}
Traceback (most recent call last):
  File "C:\Users\arjun\AppData\Local\Temp\ipykernel_33584\3407452389.py", line 15, in execute_standing_instruction
    raise InvalidTransactionError(
        f"[{account_ref}] standing instruction amount must be positive"
    )
InvalidTransactionError: [ACC-1035] standing instruction amount must be positive


REJECTED (already logged)-> [ACC-1035] standing instruction amount must be positive


### Q36. Return a structured failure result for banking case 1; use sample reference ACC-1036 and explain the result.

In [ ]:
def try_close_account(account_ref, balance):
    try:
        if balance != 0:
            raise InvalidTransactionError(f"[{account_ref}] balance must be zero to close, got {balance}")
        return {"ok": True, "account_ref": account_ref, "error": None}
    except InvalidTransactionError as exc:
        return {"ok": False, "account_ref": account_ref, "error": str(exc)}


for balance in [0, 250]:
    result = try_close_account("ACC-1036", balance)
    print(result)



{'ok': True, 'account_ref': 'ACC-1036', 'error': None}


{'ok': False, 'account_ref': 'ACC-1036', 'error': '[ACC-1036] balance must be zero to close, got 250'}


### Q37. Validate a nested beneficiary record for banking case 1; use sample reference ACC-1037 and explain the result.

In [ ]:
def validate_beneficiary(record, account_ref="ACC-1037"):
    try:
        name = record["beneficiary"]["name"]
        ifsc = record["beneficiary"]["bank"]["ifsc"]
    except KeyError as exc:
        raise InvalidTransactionError(
            f"[{account_ref}] beneficiary record missing required field: {exc}"
        ) from exc
    if not name or len(ifsc) != 11:
        raise InvalidTransactionError(f"[{account_ref}] beneficiary name/IFSC format invalid")
    return {"name": name, "ifsc": ifsc}


good = {"beneficiary": {"name": "R. Iyer", "bank": {"ifsc": "HDFC0001234"}}}
bad = {"beneficiary": {"name": "R. Iyer"}}
for record in [good, bad]:
    try:
        print(f"Valid beneficiary for ACC-1037: {validate_beneficiary(record)}")
    except InvalidTransactionError as e:
        print(f"REJECTED-> {e}")



Valid beneficiary for ACC-1037: {'name': 'R. Iyer', 'ifsc': 'HDFC0001234'}
REJECTED-> [ACC-1037] beneficiary record missing required field: 'bank'


### Q38. Use a context manager for an audit file for banking case 1; use sample reference ACC-1038 and explain the result.

In [ ]:
from contextlib import contextmanager

@contextmanager
def audit_entry(account_ref="ACC-1038"):
    print(f"[{account_ref}] AUDIT: begin")
    try:
        yield
    except Exception as exc:
        print(f"[{account_ref}] AUDIT: operation failed -> {exc}")
        raise
    finally:
        print(f"[{account_ref}] AUDIT: end")


try:
    with audit_entry():
        raise InvalidTransactionError("simulated posting failure")
except InvalidTransactionError:
    print("Caller saw the exception after audit trail was written")


[ACC-1038] AUDIT: begin
[ACC-1038] AUDIT: operation failed -> simulated posting failure
[ACC-1038] AUDIT: end
Caller saw the exception after audit trail was written


### Q39. Separate validation and recovery for banking case 1; use sample reference ACC-1039 and explain the result.

In [ ]:
def validate_transfer(amount, balance):
    """Pure validation: only raises, never attempts recovery."""
    if amount <= 0:
        raise ValueError("transfer amount must be positive")
    if amount > balance:
        raise InsufficientFundsError("transfer exceeds available balance")


def transfer(amount, balance, account_ref="ACC-1039"):
    """Recovery/handling lives here, separate from validation rules above."""
    try:
        validate_transfer(amount, balance)
    except ValueError as exc:
        print(f"[{account_ref}] input problem, transfer cancelled: {exc}")
        return balance
    except InsufficientFundsError as exc:
        print(f"[{account_ref}] funds problem, transfer cancelled: {exc}")
        return balance
    return balance - amount


print("Balance after attempt 1:", transfer(-10, 500))
print("Balance after attempt 2:", transfer(5000, 500))
print("Balance after attempt 3:", transfer(200, 500))




[ACC-1039] input problem, transfer cancelled: transfer amount must be positive
Balance after attempt 1: 500
[ACC-1039] funds problem, transfer cancelled: transfer exceeds available balance
Balance after attempt 2: 500
Balance after attempt 3: 300


### Q40. Collect all kyc validation errors for banking case 1; use sample reference ACC-1040 and explain the result.

In [ ]:
def validate_kyc(document, account_ref="ACC-1040"):
    problems = []
    if not document.get("pan"):
        problems.append("PAN is missing")
    if not document.get("address_proof"):
        problems.append("address proof is missing")
    if document.get("dob") and document["dob"] > "2020-01-01":
        problems.append("date of birth implies account holder is a minor")
    if problems:
        raise KYCValidationError(account_ref, problems)
    return True


try:
    validate_kyc({"pan": None, "address_proof": None, "dob": "2021-05-01"})
except KYCValidationError as e:
    print(f"REJECTED-> {e}")
    print("  individual problems:", e.problems)
.


REJECTED-> [ACC-1040] KYC failed: PAN is missing; address proof is missing; date of birth implies account holder is a minor
  individual problems: ['PAN is missing', 'address proof is missing', 'date of birth implies account holder is a minor']


### Q41. Create insufficientfundserror for banking case 2; use sample reference ACC-1041 and explain the result.

In [ ]:
def withdraw(balance, amount, account_ref="ACC-1041", min_balance=0):
    if amount > balance - min_balance:
        raise InsufficientFundsError(
            f"[{account_ref}] Cannot withdraw {amount}: balance {balance} "
            f"would fall below minimum {min_balance}"
        )
    return balance - amount


try:
    new_balance = withdraw(1000, 1500)
except InsufficientFundsError as e:
    print(f"REJECTED-> {e}")
else:
    print(f"New balance for ACC-1041: {new_balance}")



REJECTED-> [ACC-1041] Cannot withdraw 1500: balance 1000 would fall below minimum 0


### Q42. Chain invalidtransactionerror from valueerror for banking case 2; use sample reference ACC-1042 and explain the result.

In [ ]:
def build_transaction(raw_amount, account_ref="ACC-1042"):
    try:
        amount = float(raw_amount)
        if amount <= 0:
            raise ValueError("amount must be positive")
    except ValueError as exc:
        raise InvalidTransactionError(f"[{account_ref}] Could not build transaction") from exc
    return {"account_ref": account_ref, "amount": amount}


try:
    build_transaction("-5")
except InvalidTransactionError as e:
    print(f"REJECTED-> {e}")
    print(f"  caused by: {e.__cause__!r}")




REJECTED-> [ACC-1042] Could not build transaction
  caused by: ValueError('amount must be positive')


### Q43. Process transfers while isolating bad rows for banking case 2; use sample reference ACC-1043 and explain the result.

In [ ]:
def process_transfer_batch(rows, account_ref="ACC-1043"):
    successes, failures = [], []
    for i, row in enumerate(rows):
        try:
            amount = float(row["amount"])
            if amount <= 0:
                raise ValueError("non-positive transfer amount")
            successes.append({"row": i, "amount": amount})
        except (KeyError, ValueError) as exc:
            failures.append({"row": i, "error": str(exc)})
    return successes, failures


rows = [{"amount": "100"}, {"amount": "-5"}, {"note": "missing amount"}, {"amount": "250"}]
ok, bad = process_transfer_batch(rows)
print(f"ACC-1043: {len(ok)} succeeded, {len(bad)} failed")
for f in bad:
    print(f"  REJECTED row {f['row']}-> {f['error']}")



ACC-1043: 2 succeeded, 2 failed
  REJECTED row 1-> non-positive transfer amount
  REJECTED row 2-> 'amount'


### Q44. Retry a transient payment gateway for banking case 2; use sample reference ACC-1044 and explain the result.

In [ ]:
import random

def call_payment_gateway(account_ref="ACC-1044", _attempts_before_success=2, _state={"n": 0}):
    _state["n"] += 1
    if _state["n"] <= _attempts_before_success:
        raise TransientGatewayError(f"[{account_ref}] gateway timeout (attempt {_state['n']})")
    return f"payment-confirmed-{account_ref}"


def call_with_retry(fn, max_attempts=3):
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except TransientGatewayError as exc:
            last_exc = exc
            print(f"  retrying after: {exc}")
    raise last_exc


print(call_with_retry(call_payment_gateway))




  retrying after: [ACC-1044] gateway timeout (attempt 1)
  retrying after: [ACC-1044] gateway timeout (attempt 2)
payment-confirmed-ACC-1044


### Q45. Log failed standing instructions for banking case 2; use sample reference ACC-1045 and explain the result.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("ACC-1045")

def execute_standing_instruction(instruction, account_ref="ACC-1045"):
    try:
        if instruction["amount"] <= 0:
            raise InvalidTransactionError(f"[{account_ref}] standing instruction amount must be positive")
        return f"executed {instruction['amount']} for {account_ref}"
    except InvalidTransactionError:
        logger.error("Standing instruction failed for %s: %s", account_ref, instruction, exc_info=True)
        raise


try:
    execute_standing_instruction({"amount": -100})
except InvalidTransactionError as e:
    print(f"REJECTED (already logged)-> {e}")




ERROR: Standing instruction failed for ACC-1045: {'amount': -100}
Traceback (most recent call last):
  File "/tmp/ipykernel_544/1989614648.py", line 8, in execute_standing_instruction
    raise InvalidTransactionError(f"[{account_ref}] standing instruction amount must be positive")
InvalidTransactionError: [ACC-1045] standing instruction amount must be positive


REJECTED (already logged)-> [ACC-1045] standing instruction amount must be positive


### Q46. Return a structured failure result for banking case 2; use sample reference ACC-1046 and explain the result.

In [ ]:
def try_close_account(account_ref, balance):
    try:
        if balance != 0:
            raise InvalidTransactionError(f"[{account_ref}] balance must be zero to close, got {balance}")
        return {"ok": True, "account_ref": account_ref, "error": None}
    except InvalidTransactionError as exc:
        return {"ok": False, "account_ref": account_ref, "error": str(exc)}


for balance in [0, 250]:
    result = try_close_account("ACC-1046", balance)
    print(result)




{'ok': True, 'account_ref': 'ACC-1046', 'error': None}
{'ok': False, 'account_ref': 'ACC-1046', 'error': '[ACC-1046] balance must be zero to close, got 250'}


### Q47. Validate a nested beneficiary record for banking case 2; use sample reference ACC-1047 and explain the result.

In [ ]:
def validate_beneficiary(record, account_ref="ACC-1047"):
    try:
        name = record["beneficiary"]["name"]
        ifsc = record["beneficiary"]["bank"]["ifsc"]
    except KeyError as exc:
        raise InvalidTransactionError(
            f"[{account_ref}] beneficiary record missing required field: {exc}"
        ) from exc
    if not name or len(ifsc) != 11:
        raise InvalidTransactionError(f"[{account_ref}] beneficiary name/IFSC format invalid")
    return {"name": name, "ifsc": ifsc}


good = {"beneficiary": {"name": "R. Iyer", "bank": {"ifsc": "HDFC0001234"}}}
bad = {"beneficiary": {"name": "R. Iyer"}}
for record in [good, bad]:
    try:
        print(f"Valid beneficiary for ACC-1047: {validate_beneficiary(record)}")
    except InvalidTransactionError as e:
        print(f"REJECTED-> {e}")



Valid beneficiary for ACC-1047: {'name': 'R. Iyer', 'ifsc': 'HDFC0001234'}
REJECTED-> [ACC-1047] beneficiary record missing required field: 'bank'


### Q48. Use a context manager for an audit file for banking case 2; use sample reference ACC-1048 and explain the result.

In [ ]:
from contextlib import contextmanager

@contextmanager
def audit_entry(account_ref="ACC-1048"):
    print(f"[{account_ref}] AUDIT: begin")
    try:
        yield
    except Exception as exc:
        print(f"[{account_ref}] AUDIT: operation failed -> {exc}")
        raise
    finally:
        print(f"[{account_ref}] AUDIT: end")


try:
    with audit_entry():
        raise InvalidTransactionError("simulated posting failure")
except InvalidTransactionError:
    print("Caller saw the exception after audit trail was written")



[ACC-1048] AUDIT: begin
[ACC-1048] AUDIT: operation failed -> simulated posting failure
[ACC-1048] AUDIT: end
Caller saw the exception after audit trail was written


### Q49. Separate validation and recovery for banking case 2; use sample reference ACC-1049 and explain the result.

In [ ]:
def validate_transfer(amount, balance):
    """Pure validation: only raises, never attempts recovery."""
    if amount <= 0:
        raise ValueError("transfer amount must be positive")
    if amount > balance:
        raise InsufficientFundsError("transfer exceeds available balance")


def transfer(amount, balance, account_ref="ACC-1049"):
    """Recovery/handling lives here, separate from validation rules above."""
    try:
        validate_transfer(amount, balance)
    except ValueError as exc:
        print(f"[{account_ref}] input problem, transfer cancelled: {exc}")
        return balance
    except InsufficientFundsError as exc:
        print(f"[{account_ref}] funds problem, transfer cancelled: {exc}")
        return balance
    return balance - amount


print("Balance after attempt 1:", transfer(-10, 500))
print("Balance after attempt 2:", transfer(5000, 500))
print("Balance after attempt 3:", transfer(200, 500))




[ACC-1049] input problem, transfer cancelled: transfer amount must be positive
Balance after attempt 1: 500
[ACC-1049] funds problem, transfer cancelled: transfer exceeds available balance
Balance after attempt 2: 500
Balance after attempt 3: 300


### Q50. Collect all kyc validation errors for banking case 2; use sample reference ACC-1050 and explain the result.

In [ ]:
def validate_kyc(document, account_ref="ACC-1050"):
    problems = []
    if not document.get("pan"):
        problems.append("PAN is missing")
    if not document.get("address_proof"):
        problems.append("address proof is missing")
    if document.get("dob") and document["dob"] > "2020-01-01":
        problems.append("date of birth implies account holder is a minor")
    if problems:
        raise KYCValidationError(account_ref, problems)
    return True


try:
    validate_kyc({"pan": None, "address_proof": None, "dob": "2021-05-01"})
except KYCValidationError as e:
    print(f"REJECTED-> {e}")
    print("  individual problems:", e.problems)




REJECTED-> [ACC-1050] KYC failed: PAN is missing; address proof is missing; date of birth implies account holder is a minor
  individual problems: ['PAN is missing', 'address proof is missing', 'date of birth implies account holder is a minor']


### Q51. Create insufficientfundserror for banking case 3; use sample reference ACC-1051 and explain the result.

In [ ]:
def withdraw(balance, amount, account_ref="ACC-1051", min_balance=0):
    if amount > balance - min_balance:
        raise InsufficientFundsError(
            f"[{account_ref}] Cannot withdraw {amount}: balance {balance} "
            f"would fall below minimum {min_balance}"
        )
    return balance - amount


try:
    new_balance = withdraw(1000, 1500)
except InsufficientFundsError as e:
    print(f"REJECTED-> {e}")
else:
    print(f"New balance for ACC-1051: {new_balance}")




REJECTED-> [ACC-1051] Cannot withdraw 1500: balance 1000 would fall below minimum 0


### Q52. Chain invalidtransactionerror from valueerror for banking case 3; use sample reference ACC-1052 and explain the result.

In [ ]:
def build_transaction(raw_amount, account_ref="ACC-1052"):
    try:
        amount = float(raw_amount)
        if amount <= 0:
            raise ValueError("amount must be positive")
    except ValueError as exc:
        raise InvalidTransactionError(f"[{account_ref}] Could not build transaction") from exc
    return {"account_ref": account_ref, "amount": amount}


try:
    build_transaction("-5")
except InvalidTransactionError as e:
    print(f"REJECTED-> {e}")
    print(f"  caused by: {e.__cause__!r}")




REJECTED-> [ACC-1052] Could not build transaction
  caused by: ValueError('amount must be positive')


### Q53. Process transfers while isolating bad rows for banking case 3; use sample reference ACC-1053 and explain the result.

In [ ]:
def process_transfer_batch(rows, account_ref="ACC-1053"):
    successes, failures = [], []
    for i, row in enumerate(rows):
        try:
            amount = float(row["amount"])
            if amount <= 0:
                raise ValueError("non-positive transfer amount")
            successes.append({"row": i, "amount": amount})
        except (KeyError, ValueError) as exc:
            failures.append({"row": i, "error": str(exc)})
    return successes, failures


rows = [{"amount": "100"}, {"amount": "-5"}, {"note": "missing amount"}, {"amount": "250"}]
ok, bad = process_transfer_batch(rows)
print(f"ACC-1053: {len(ok)} succeeded, {len(bad)} failed")
for f in bad:
    print(f"  REJECTED row {f['row']}-> {f['error']}")




ACC-1053: 2 succeeded, 2 failed


  REJECTED row 1-> non-positive transfer amount
  REJECTED row 2-> 'amount'


### Q54. Retry a transient payment gateway for banking case 3; use sample reference ACC-1054 and explain the result.

In [ ]:
import random

def call_payment_gateway(account_ref="ACC-1054", _attempts_before_success=2, _state={"n": 0}):
    _state["n"] += 1
    if _state["n"] <= _attempts_before_success:
        raise TransientGatewayError(f"[{account_ref}] gateway timeout (attempt {_state['n']})")
    return f"payment-confirmed-{account_ref}"


def call_with_retry(fn, max_attempts=3):
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except TransientGatewayError as exc:
            last_exc = exc
            print(f"  retrying after: {exc}")
    raise last_exc


print(call_with_retry(call_payment_gateway))




  retrying after: [ACC-1054] gateway timeout (attempt 1)
  retrying after: [ACC-1054] gateway timeout (attempt 2)
payment-confirmed-ACC-1054


### Q55. Log failed standing instructions for banking case 3; use sample reference ACC-1055 and explain the result.

In [1]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("ACC-1055")

def execute_standing_instruction(instruction, account_ref="ACC-1055"):
    try:
        if instruction["amount"] <= 0:
            raise InvalidTransactionError(f"[{account_ref}] standing instruction amount must be positive")
        return f"executed {instruction['amount']} for {account_ref}"
    except InvalidTransactionError:
        logger.error("Standing instruction failed for %s: %s", account_ref, instruction, exc_info=True)
        raise


try:
    execute_standing_instruction({"amount": -100})
except InvalidTransactionError as e:
    print(f"REJECTED (already logged)-> {e}")



NameError: name 'InvalidTransactionError' is not defined

### Q56. Return a structured failure result for banking case 3; use sample reference ACC-1056 and explain the result.

In [ ]:
def try_close_account(account_ref, balance):
    try:
        if balance != 0:
            raise InvalidTransactionError(f"[{account_ref}] balance must be zero to close, got {balance}")
        return {"ok": True, "account_ref": account_ref, "error": None}
    except InvalidTransactionError as exc:
        return {"ok": False, "account_ref": account_ref, "error": str(exc)}


for balance in [0, 250]:
    result = try_close_account("ACC-1056", balance)
    print(result)




{'ok': True, 'account_ref': 'ACC-1056', 'error': None}


{'ok': False, 'account_ref': 'ACC-1056', 'error': '[ACC-1056] balance must be zero to close, got 250'}


### Q57. Validate a nested beneficiary record for banking case 3; use sample reference ACC-1057 and explain the result.

In [ ]:
def validate_beneficiary(record, account_ref="ACC-1057"):
    try:
        name = record["beneficiary"]["name"]
        ifsc = record["beneficiary"]["bank"]["ifsc"]
    except KeyError as exc:
        raise InvalidTransactionError(
            f"[{account_ref}] beneficiary record missing required field: {exc}"
        ) from exc
    if not name or len(ifsc) != 11:
        raise InvalidTransactionError(f"[{account_ref}] beneficiary name/IFSC format invalid")
    return {"name": name, "ifsc": ifsc}


good = {"beneficiary": {"name": "R. Iyer", "bank": {"ifsc": "HDFC0001234"}}}
bad = {"beneficiary": {"name": "R. Iyer"}}
for record in [good, bad]:
    try:
        print(f"Valid beneficiary for ACC-1057: {validate_beneficiary(record)}")
    except InvalidTransactionError as e:
        print(f"REJECTED-> {e}")


Valid beneficiary for ACC-1057: {'name': 'R. Iyer', 'ifsc': 'HDFC0001234'}
REJECTED-> [ACC-1057] beneficiary record missing required field: 'bank'


### Q58. Use a context manager for an audit file for banking case 3; use sample reference ACC-1058 and explain the result.

In [ ]:
from contextlib import contextmanager

@contextmanager
def audit_entry(account_ref="ACC-1058"):
    print(f"[{account_ref}] AUDIT: begin")
    try:
        yield
    except Exception as exc:
        print(f"[{account_ref}] AUDIT: operation failed -> {exc}")
        raise
    finally:
        print(f"[{account_ref}] AUDIT: end")


try:
    with audit_entry():
        raise InvalidTransactionError("simulated posting failure")
except InvalidTransactionError:
    print("Caller saw the exception after audit trail was written")




[ACC-1058] AUDIT: begin
[ACC-1058] AUDIT: operation failed -> simulated posting failure
[ACC-1058] AUDIT: end
Caller saw the exception after audit trail was written


### Q59. Separate validation and recovery for banking case 3; use sample reference ACC-1059 and explain the result.

In [ ]:
def validate_transfer(amount, balance):
    """Pure validation: only raises, never attempts recovery."""
    if amount <= 0:
        raise ValueError("transfer amount must be positive")
    if amount > balance:
        raise InsufficientFundsError("transfer exceeds available balance")


def transfer(amount, balance, account_ref="ACC-1059"):
    """Recovery/handling lives here, separate from validation rules above."""
    try:
        validate_transfer(amount, balance)
    except ValueError as exc:
        print(f"[{account_ref}] input problem, transfer cancelled: {exc}")
        return balance
    except InsufficientFundsError as exc:
        print(f"[{account_ref}] funds problem, transfer cancelled: {exc}")
        return balance
    return balance - amount


print("Balance after attempt 1:", transfer(-10, 500))
print("Balance after attempt 2:", transfer(5000, 500))
print("Balance after attempt 3:", transfer(200, 500))




[ACC-1059] input problem, transfer cancelled: transfer amount must be positive
Balance after attempt 1: 500
[ACC-1059] funds problem, transfer cancelled: transfer exceeds available balance
Balance after attempt 2: 500
Balance after attempt 3: 300


### Q60. Collect all kyc validation errors for banking case 3; use sample reference ACC-1060 and explain the result.

In [ ]:
def validate_kyc(document, account_ref="ACC-1060"):
    problems = []
    if not document.get("pan"):
        problems.append("PAN is missing")
    if not document.get("address_proof"):
        problems.append("address proof is missing")
    if document.get("dob") and document["dob"] > "2020-01-01":
        problems.append("date of birth implies account holder is a minor")
    if problems:
        raise KYCValidationError(account_ref, problems)
    return True


try:
    validate_kyc({"pan": None, "address_proof": None, "dob": "2021-05-01"})
except KYCValidationError as e:
    print(f"REJECTED-> {e}")
    print("  individual problems:", e.problems)




REJECTED-> [ACC-1060] KYC failed: PAN is missing; address proof is missing; date of birth implies account holder is a minor
  individual problems: ['PAN is missing', 'address proof is missing', 'date of birth implies account holder is a minor']


### Q61. Create insufficientfundserror for banking case 4; use sample reference ACC-1061 and explain the result.

In [ ]:
def withdraw(balance, amount, account_ref="ACC-1061", min_balance=0):
    if amount > balance - min_balance:
        raise InsufficientFundsError(
            f"[{account_ref}] Cannot withdraw {amount}: balance {balance} "
            f"would fall below minimum {min_balance}"
        )
    return balance - amount


try:
    new_balance = withdraw(1000, 1500)
except InsufficientFundsError as e:
    print(f"REJECTED-> {e}")
else:
    print(f"New balance for ACC-1061: {new_balance}")



REJECTED-> [ACC-1061] Cannot withdraw 1500: balance 1000 would fall below minimum 0


### Q62. Chain invalidtransactionerror from valueerror for banking case 4; use sample reference ACC-1062 and explain the result.

In [ ]:
def build_transaction(raw_amount, account_ref="ACC-1062"):
    try:
        amount = float(raw_amount)
        if amount <= 0:
            raise ValueError("amount must be positive")
    except ValueError as exc:
        raise InvalidTransactionError(f"[{account_ref}] Could not build transaction") from exc
    return {"account_ref": account_ref, "amount": amount}


try:
    build_transaction("-5")
except InvalidTransactionError as e:
    print(f"REJECTED-> {e}")
    print(f"  caused by: {e.__cause__!r}")




REJECTED-> [ACC-1062] Could not build transaction
  caused by: ValueError('amount must be positive')


### Q63. Process transfers while isolating bad rows for banking case 4; use sample reference ACC-1063 and explain the result.

In [ ]:
def process_transfer_batch(rows, account_ref="ACC-1063"):
    successes, failures = [], []
    for i, row in enumerate(rows):
        try:
            amount = float(row["amount"])
            if amount <= 0:
                raise ValueError("non-positive transfer amount")
            successes.append({"row": i, "amount": amount})
        except (KeyError, ValueError) as exc:
            failures.append({"row": i, "error": str(exc)})
    return successes, failures


rows = [{"amount": "100"}, {"amount": "-5"}, {"note": "missing amount"}, {"amount": "250"}]
ok, bad = process_transfer_batch(rows)
print(f"ACC-1063: {len(ok)} succeeded, {len(bad)} failed")
for f in bad:
    print(f"  REJECTED row {f['row']}-> {f['error']}")




ACC-1063: 2 succeeded, 2 failed


  REJECTED row 1-> non-positive transfer amount
  REJECTED row 2-> 'amount'


### Q64. Retry a transient payment gateway for banking case 4; use sample reference ACC-1064 and explain the result.

In [ ]:
import random

def call_payment_gateway(account_ref="ACC-1064", _attempts_before_success=2, _state={"n": 0}):
    _state["n"] += 1
    if _state["n"] <= _attempts_before_success:
        raise TransientGatewayError(f"[{account_ref}] gateway timeout (attempt {_state['n']})")
    return f"payment-confirmed-{account_ref}"


def call_with_retry(fn, max_attempts=3):
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except TransientGatewayError as exc:
            last_exc = exc
            print(f"  retrying after: {exc}")
    raise last_exc


print(call_with_retry(call_payment_gateway))



  retrying after: [ACC-1064] gateway timeout (attempt 1)
  retrying after: [ACC-1064] gateway timeout (attempt 2)
payment-confirmed-ACC-1064


### Q65. Log failed standing instructions for banking case 4; use sample reference ACC-1065 and explain the result.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("ACC-1065")

def execute_standing_instruction(instruction, account_ref="ACC-1065"):
    try:
        if instruction["amount"] <= 0:
            raise InvalidTransactionError(f"[{account_ref}] standing instruction amount must be positive")
        return f"executed {instruction['amount']} for {account_ref}"
    except InvalidTransactionError:
        logger.error("Standing instruction failed for %s: %s", account_ref, instruction, exc_info=True)
        raise


try:
    execute_standing_instruction({"amount": -100})
except InvalidTransactionError as e:
    print(f"REJECTED (already logged)-> {e}")



ERROR: Standing instruction failed for ACC-1065: {'amount': -100}
Traceback (most recent call last):
  File "/tmp/ipykernel_544/2071738326.py", line 8, in execute_standing_instruction
    raise InvalidTransactionError(f"[{account_ref}] standing instruction amount must be positive")
InvalidTransactionError: [ACC-1065] standing instruction amount must be positive


REJECTED (already logged)-> [ACC-1065] standing instruction amount must be positive


### Q66. Return a structured failure result for banking case 4; use sample reference ACC-1066 and explain the result.

In [ ]:
def try_close_account(account_ref, balance):
    try:
        if balance != 0:
            raise InvalidTransactionError(f"[{account_ref}] balance must be zero to close, got {balance}")
        return {"ok": True, "account_ref": account_ref, "error": None}
    except InvalidTransactionError as exc:
        return {"ok": False, "account_ref": account_ref, "error": str(exc)}


for balance in [0, 250]:
    result = try_close_account("ACC-1066", balance)
    print(result)



{'ok': True, 'account_ref': 'ACC-1066', 'error': None}


{'ok': False, 'account_ref': 'ACC-1066', 'error': '[ACC-1066] balance must be zero to close, got 250'}


### Q67. Validate a nested beneficiary record for banking case 4; use sample reference ACC-1067 and explain the result.

In [ ]:
def validate_beneficiary(record, account_ref="ACC-1067"):
    try:
        name = record["beneficiary"]["name"]
        ifsc = record["beneficiary"]["bank"]["ifsc"]
    except KeyError as exc:
        raise InvalidTransactionError(
            f"[{account_ref}] beneficiary record missing required field: {exc}"
        ) from exc
    if not name or len(ifsc) != 11:
        raise InvalidTransactionError(f"[{account_ref}] beneficiary name/IFSC format invalid")
    return {"name": name, "ifsc": ifsc}


good = {"beneficiary": {"name": "R. Iyer", "bank": {"ifsc": "HDFC0001234"}}}
bad = {"beneficiary": {"name": "R. Iyer"}}
for record in [good, bad]:
    try:
        print(f"Valid beneficiary for ACC-1067: {validate_beneficiary(record)}")
    except InvalidTransactionError as e:
        print(f"REJECTED-> {e}")


Valid beneficiary for ACC-1067: {'name': 'R. Iyer', 'ifsc': 'HDFC0001234'}


REJECTED-> [ACC-1067] beneficiary record missing required field: 'bank'


### Q68. Use a context manager for an audit file for banking case 4; use sample reference ACC-1068 and explain the result.

In [ ]:
from contextlib import contextmanager

@contextmanager
def audit_entry(account_ref="ACC-1068"):
    print(f"[{account_ref}] AUDIT: begin")
    try:
        yield
    except Exception as exc:
        print(f"[{account_ref}] AUDIT: operation failed -> {exc}")
        raise
    finally:
        print(f"[{account_ref}] AUDIT: end")


try:
    with audit_entry():
        raise InvalidTransactionError("simulated posting failure")
except InvalidTransactionError:
    print("Caller saw the exception after audit trail was written")




[ACC-1068] AUDIT: begin
[ACC-1068] AUDIT: operation failed -> simulated posting failure
[ACC-1068] AUDIT: end
Caller saw the exception after audit trail was written


### Q69. Separate validation and recovery for banking case 4; use sample reference ACC-1069 and explain the result.

In [ ]:
def validate_transfer(amount, balance):
    """Pure validation: only raises, never attempts recovery."""
    if amount <= 0:
        raise ValueError("transfer amount must be positive")
    if amount > balance:
        raise InsufficientFundsError("transfer exceeds available balance")


def transfer(amount, balance, account_ref="ACC-1069"):
    """Recovery/handling lives here, separate from validation rules above."""
    try:
        validate_transfer(amount, balance)
    except ValueError as exc:
        print(f"[{account_ref}] input problem, transfer cancelled: {exc}")
        return balance
    except InsufficientFundsError as exc:
        print(f"[{account_ref}] funds problem, transfer cancelled: {exc}")
        return balance
    return balance - amount


print("Balance after attempt 1:", transfer(-10, 500))
print("Balance after attempt 2:", transfer(5000, 500))
print("Balance after attempt 3:", transfer(200, 500))



[ACC-1069] input problem, transfer cancelled: transfer amount must be positive
Balance after attempt 1: 500
[ACC-1069] funds problem, transfer cancelled: transfer exceeds available balance
Balance after attempt 2: 500
Balance after attempt 3: 300


### Q70. Collect all kyc validation errors for banking case 4; use sample reference ACC-1070 and explain the result.

In [ ]:
def validate_kyc(document, account_ref="ACC-1070"):
    problems = []
    if not document.get("pan"):
        problems.append("PAN is missing")
    if not document.get("address_proof"):
        problems.append("address proof is missing")
    if document.get("dob") and document["dob"] > "2020-01-01":
        problems.append("date of birth implies account holder is a minor")
    if problems:
        raise KYCValidationError(account_ref, problems)
    return True


try:
    validate_kyc({"pan": None, "address_proof": None, "dob": "2021-05-01"})
except KYCValidationError as e:
    print(f"REJECTED-> {e}")
    print("  individual problems:", e.problems)




REJECTED-> [ACC-1070] KYC failed: PAN is missing; address proof is missing; date of birth implies account holder is a minor
  individual problems: ['PAN is missing', 'address proof is missing', 'date of birth implies account holder is a minor']


## Difficult — Architecture, edge cases, and end-to-end banking systems (Q71–Q100)

### Q71. Design a banking exception hierarchy for banking case 1; use sample reference ACC-1071 and explain the result.

In [ ]:
class AccountError(BankingError):
    """Problems tied to a specific account (not found, frozen, closed)."""

class TransactionError(BankingError):
    """Problems tied to a specific transaction attempt."""

class ComplianceError(BankingError):
    """KYC / regulatory / fraud-monitoring problems."""

class FrozenAccountError(AccountError):
    pass

class DuplicateTransactionError(TransactionError):
    pass


def post_transaction(account_ref, frozen, amount, seen_ids, txn_id):
    if frozen:
        raise FrozenAccountError(f"[{account_ref}] account is frozen")
    if txn_id in seen_ids:
        raise DuplicateTransactionError(f"[{account_ref}] transaction {txn_id} already processed")
    seen_ids.add(txn_id)
    return f"posted {amount} for {account_ref}"


seen = set()
for frozen, amount, txn_id in [(True, 100, "T1"), (False, 100, "T1"), (False, 100, "T2")]:
    try:
        print(post_transaction("ACC-1071", frozen, amount, seen, txn_id))
    except BankingError as e:
        # Catching the common base still tells us the concrete subtype via type(e).
        print(f"REJECTED ({type(e).__name__})-> {e}")



REJECTED (FrozenAccountError)-> [ACC-1071] account is frozen
posted 100 for ACC-1071
posted 100 for ACC-1071


### Q72. Make a transfer rollback-safe for banking case 1; use sample reference ACC-1072 and explain the result.

In [ ]:
def transfer_funds(ledger, from_ref, to_ref, amount):
    original_from = ledger[from_ref]
    original_to = ledger[to_ref]
    try:
        if ledger[from_ref] < amount:
            raise InsufficientFundsError(f"{from_ref} has insufficient funds for {amount}")
        ledger[from_ref] -= amount
        ledger[to_ref] += amount
        if to_ref not in ledger:  # simulated post-debit failure e.g. bad destination
            raise AccountNotFoundError(f"destination {to_ref} vanished mid-transfer")
        return ledger
    except (InsufficientFundsError, AccountNotFoundError):
        # Roll both legs back to their pre-transfer values before re-raising.
        ledger[from_ref] = original_from
        ledger[to_ref] = original_to
        raise


ledger = {"ACC-1072": 1000, "ACC-DEST": 500}
try:
    transfer_funds(ledger, "ACC-1072", "ACC-DEST", 5000)
except InsufficientFundsError as e:
    print(f"REJECTED-> {e}")
print("Ledger unchanged after rollback:", ledger)




REJECTED-> ACC-1072 has insufficient funds for 5000


Ledger unchanged after rollback: {'ACC-1072': 1000, 'ACC-DEST': 500}


### Q73. Preserve causes across service layers for banking case 1; use sample reference ACC-1073 and explain the result.

In [ ]:
# Layer 1: low-level storage
def storage_read(key):
    raise KeyError(key)

# Layer 2: repository, translates storage errors to domain errors
def repo_get_account(account_ref):
    try:
        return storage_read(account_ref)
    except KeyError as exc:
        raise AccountNotFoundError(f"repository: {account_ref} not found") from exc

# Layer 3: service, translates repository errors to API-facing errors
def service_get_balance(account_ref="ACC-1073"):
    try:
        return repo_get_account(account_ref)
    except AccountNotFoundError as exc:
        raise InvalidTransactionError(f"service: cannot fetch balance for {account_ref}") from exc


try:
    service_get_balance()
except InvalidTransactionError as e:
    chain = []
    cur = e
    while cur is not None:
        chain.append(f"{type(cur).__name__}: {cur}")
        cur = cur.__cause__
    print("Full cause chain (top-level first):")
    for line in chain:
        print(" ->", line)



Full cause chain (top-level first):


 -> InvalidTransactionError: service: cannot fetch balance for ACC-1073
 -> AccountNotFoundError: repository: ACC-1073 not found
 -> KeyError: 'ACC-1073'


### Q74. Implement bounded exponential retry for banking case 1; use sample reference ACC-1074 and explain the result.

In [ ]:
import time

def call_gateway_flaky(account_ref="ACC-1074", _state={"n": 0}):
    _state["n"] += 1
    if _state["n"] < 3:
        raise TransientGatewayError(f"[{account_ref}] transient failure #{_state['n']}")
    return f"settled for {account_ref}"


def retry_with_backoff(fn, max_attempts=4, base_delay=0.01):
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except TransientGatewayError as exc:
            if attempt == max_attempts:
                raise TransientGatewayError(
                    f"giving up after {max_attempts} attempts"
                ) from exc
            delay = base_delay * (2 ** (attempt - 1))
            print(f"  attempt {attempt} failed ({exc}); backing off {delay:.3f}s")
            time.sleep(delay)


print(retry_with_backoff(call_gateway_flaky))




  attempt 1 failed ([ACC-1074] transient failure #1); backing off 0.010s

  attempt 2 failed ([ACC-1074] transient failure #2); backing off 0.020s


settled for ACC-1074


### Q75. Create a transaction context manager for banking case 1; use sample reference ACC-1075 and explain the result.

In [ ]:
class transaction_scope:
    """Commits ledger changes only if the block completes without error."""

    def __init__(self, ledger, account_ref="ACC-1075"):
        self.ledger = ledger
        self.account_ref = account_ref
        self.snapshot = None

    def __enter__(self):
        self.snapshot = dict(self.ledger)
        return self.ledger

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            self.ledger.clear()
            self.ledger.update(self.snapshot)
            print(f"[{self.account_ref}] transaction rolled back due to {exc_type.__name__}: {exc_val}")
            return False  # do not suppress -- caller still sees the exception
        print(f"[{self.account_ref}] transaction committed")
        return False


ledger = {"ACC-1075": 1000}
try:
    with transaction_scope(ledger) as tx:
        tx["ACC-1075"] -= 200
        raise InvalidTransactionError("downstream validation failed")
except InvalidTransactionError as e:
    print(f"REJECTED-> {e}")
print("Ledger after failed transaction:", ledger)



[ACC-1075] transaction rolled back due to InvalidTransactionError: downstream validation failed
REJECTED-> downstream validation failed
Ledger after failed transaction: {'ACC-1075': 1000}


### Q76. Build a dead-letter queue for payment events for banking case 1; use sample reference ACC-1076 and explain the result.

In [ ]:
dead_letter_queue = []

def handle_payment_event(event, account_ref="ACC-1076"):
    try:
        if event["amount"] <= 0:
            raise InvalidTransactionError(f"non-positive amount in event {event}")
        if event.get("gateway_down"):
            raise TransientGatewayError(f"gateway unavailable for event {event}")
        return f"processed {event['amount']} for {account_ref}"
    except InvalidTransactionError as exc:
        # Not retryable -- permanently bad event, park it for manual review.
        dead_letter_queue.append({"event": event, "reason": str(exc), "retryable": False})
        return None
    except TransientGatewayError as exc:
        # Retryable -- park it too, but flagged so a separate worker can retry later.
        dead_letter_queue.append({"event": event, "reason": str(exc), "retryable": True})
        return None


events = [{"amount": 100}, {"amount": -5}, {"amount": 50, "gateway_down": True}]
for e in events:
    print("result:", handle_payment_event(e))

print(f"Dead-letter queue for ACC-1076 has {len(dead_letter_queue)} entries:")
for entry in dead_letter_queue:
    print(" ", entry)



result: processed 100 for ACC-1076
result: None
result: None
Dead-letter queue for ACC-1076 has 2 entries:
  {'event': {'amount': -5}, 'reason': "non-positive amount in event {'amount': -5}", 'retryable': False}
  {'event': {'amount': 50, 'gateway_down': True}, 'reason': "gateway unavailable for event {'amount': 50, 'gateway_down': True}", 'retryable': True}


### Q77. Make batch settlement failure-isolated for banking case 1; use sample reference ACC-1077 and explain the result.

In [ ]:
def settle_batch(entries, account_ref="ACC-1077"):
    report = {"settled": [], "failed": []}
    for entry in entries:
        try:
            amount = float(entry["amount"])
            if amount <= 0:
                raise InvalidTransactionError("settlement amount must be positive")
            report["settled"].append({"id": entry["id"], "amount": amount})
        except (KeyError, ValueError, InvalidTransactionError) as exc:
            report["failed"].append({"id": entry.get("id", "<unknown>"), "error": str(exc)})
            continue  # move on to the next entry regardless of this failure
    return report


entries = [
    {"id": "S1", "amount": "100"},
    {"id": "S2", "amount": "-20"},
    {"amount": "50"},          # missing id
    {"id": "S4", "amount": "abc"},
    {"id": "S5", "amount": "75"},
]
report = settle_batch(entries)
print(f"ACC-1077 settlement report: {len(report['settled'])} settled, {len(report['failed'])} failed")
for f in report["failed"]:
    print("  failed:", f)




ACC-1077 settlement report: 2 settled, 3 failed
  failed: {'id': 'S2', 'error': 'settlement amount must be positive'}
  failed: {'id': '<unknown>', 'error': "'id'"}
  failed: {'id': 'S4', 'error': "could not convert string to float: 'abc'"}


### Q78. Report exact paths in nested statements for banking case 1; use sample reference ACC-1078 and explain the result.

In [ ]:
def validate_application(data, account_ref="ACC-1078", path=""):
    """Recursively validate a nested loan-application dict, reporting the exact field path."""
    errors = []
    for key, value in data.items():
        current_path = f"{path}.{key}" if path else key
        if isinstance(value, dict):
            errors.extend(validate_application(value, account_ref, current_path))
        elif value in (None, ""):
            errors.append(f"[{account_ref}] field '{current_path}' is required")
    return errors


application = {
    "applicant": {"name": "A. Rao", "employment": {"employer": "", "years": 3}},
    "loan": {"amount": 200000, "purpose": None},
}

try:
    errors = validate_application(application)
    if errors:
        raise InvalidTransactionError("; ".join(errors))
    print(f"[ACC-1078] application valid")
except InvalidTransactionError as e:
    print(f"REJECTED-> {e}")



REJECTED-> [ACC-1078] field 'applicant.employment.employer' is required; [ACC-1078] field 'loan.purpose' is required

### Q79. Design idempotent recovery after timeout for banking case 1; use sample reference ACC-1079 and explain the result.

In [ ]:
processed_idempotency_keys = {}

def submit_payment(idempotency_key, amount, account_ref="ACC-1079", _simulate_timeout={"n": 0}):
    if idempotency_key in processed_idempotency_keys:
        # We've seen this exact request before -- return the stored result instead of
        # re-charging the account a second time.
        print(f"[{account_ref}] duplicate submission detected, returning cached result")
        return processed_idempotency_keys[idempotency_key]

    _simulate_timeout["n"] += 1
    if _simulate_timeout["n"] == 1:
        # Simulate: the request reached the server and would have succeeded, but the
        # client never got the response (network timeout) and does not know the outcome.
        raise TransientGatewayError(f"[{account_ref}] response timed out, outcome unknown")

    result = f"charged {amount} for {account_ref} (key={idempotency_key})"
    processed_idempotency_keys[idempotency_key] = result
    return result


key = "idem-ACC-1079-001"
try:
    submit_payment(key, 500)
except TransientGatewayError as e:
    print(f"REJECTED-> {e}")
    print("Client retries with the SAME idempotency key:")
    print(submit_payment(key, 500))
print("Client retries again (already processed):")
print(submit_payment(key, 500))


REJECTED-> [ACC-1079] response timed out, outcome unknown
Client retries with the SAME idempotency key:
charged 500 for ACC-1079 (key=idem-ACC-1079-001)
Client retries again (already processed):
[ACC-1079] duplicate submission detected, returning cached result
charged 500 for ACC-1079 (key=idem-ACC-1079-001)


### Q80. Build an exception-safe reconciliation pipeline for banking case 1; use sample reference ACC-1080 and explain the result.

In [ ]:
def fetch_bank_statement(account_ref):
    return [{"id": "T1", "amount": 100}, {"id": "T2", "amount": 250}, {"id": "T3", "amount": -50}]

def fetch_internal_ledger(account_ref):
    return [{"id": "T1", "amount": 100}, {"id": "T2", "amount": 260}]  # T2 mismatched, T3 missing


def reconcile(account_ref="ACC-1080"):
    stage_results = {"fetched_statement": None, "fetched_ledger": None, "diffs": None}
    try:
        stage_results["fetched_statement"] = fetch_bank_statement(account_ref)
        stage_results["fetched_ledger"] = fetch_internal_ledger(account_ref)
    except Exception as exc:
        raise InvalidTransactionError(f"[{account_ref}] reconciliation aborted: could not fetch data") from exc

    diffs = []
    ledger_by_id = {row["id"]: row["amount"] for row in stage_results["fetched_ledger"]}
    for row in stage_results["fetched_statement"]:
        try:
            ledger_amount = ledger_by_id[row["id"]]
            if ledger_amount != row["amount"]:
                diffs.append(f"{row['id']}: statement={row['amount']} ledger={ledger_amount}")
        except KeyError:
            diffs.append(f"{row['id']}: present in statement, missing from ledger")
    stage_results["diffs"] = diffs
    return stage_results


result = reconcile()
print(f"Reconciliation for ACC-1080:")
print(f"  statement rows: {len(result['fetched_statement'])}")
print(f"  ledger rows:    {len(result['fetched_ledger'])}")
print(f"  discrepancies:  {len(result['diffs'])}")
for d in result["diffs"]:
    print("   -", d)




Reconciliation for ACC-1080:
  statement rows: 3
  ledger rows:    2
  discrepancies:  2
   - T2: statement=250 ledger=260
   - T3: present in statement, missing from ledger


### Q81. Design a banking exception hierarchy for banking case 2; use sample reference ACC-1081 and explain the result.

In [ ]:
class AccountError(BankingError):
    """Problems tied to a specific account (not found, frozen, closed)."""

class TransactionError(BankingError):
    """Problems tied to a specific transaction attempt."""

class ComplianceError(BankingError):
    """KYC / regulatory / fraud-monitoring problems."""

class FrozenAccountError(AccountError):
    pass

class DuplicateTransactionError(TransactionError):
    pass


def post_transaction(account_ref, frozen, amount, seen_ids, txn_id):
    if frozen:
        raise FrozenAccountError(f"[{account_ref}] account is frozen")
    if txn_id in seen_ids:
        raise DuplicateTransactionError(f"[{account_ref}] transaction {txn_id} already processed")
    seen_ids.add(txn_id)
    return f"posted {amount} for {account_ref}"


seen = set()
for frozen, amount, txn_id in [(True, 100, "T1"), (False, 100, "T1"), (False, 100, "T2")]:
    try:
        print(post_transaction("ACC-1081", frozen, amount, seen, txn_id))
    except BankingError as e:
        # Catching the common base still tells us the concrete subtype via type(e).
        print(f"REJECTED ({type(e).__name__})-> {e}")



REJECTED (FrozenAccountError)-> [ACC-1081] account is frozen
posted 100 for ACC-1081
posted 100 for ACC-1081


### Q82. Make a transfer rollback-safe for banking case 2; use sample reference ACC-1082 and explain the result.

In [ ]:
def transfer_funds(ledger, from_ref, to_ref, amount):
    original_from = ledger[from_ref]
    original_to = ledger[to_ref]
    try:
        if ledger[from_ref] < amount:
            raise InsufficientFundsError(f"{from_ref} has insufficient funds for {amount}")
        ledger[from_ref] -= amount
        ledger[to_ref] += amount
        if to_ref not in ledger:  # simulated post-debit failure e.g. bad destination
            raise AccountNotFoundError(f"destination {to_ref} vanished mid-transfer")
        return ledger
    except (InsufficientFundsError, AccountNotFoundError):
        # Roll both legs back to their pre-transfer values before re-raising.
        ledger[from_ref] = original_from
        ledger[to_ref] = original_to
        raise


ledger = {"ACC-1082": 1000, "ACC-DEST": 500}
try:
    transfer_funds(ledger, "ACC-1082", "ACC-DEST", 5000)
except InsufficientFundsError as e:
    print(f"REJECTED-> {e}")
print("Ledger unchanged after rollback:", ledger)




REJECTED-> ACC-1082 has insufficient funds for 5000
Ledger unchanged after rollback: {'ACC-1082': 1000, 'ACC-DEST': 500}


### Q83. Preserve causes across service layers for banking case 2; use sample reference ACC-1083 and explain the result.

In [ ]:
# Layer 1: low-level storage
def storage_read(key):
    raise KeyError(key)

# Layer 2: repository, translates storage errors to domain errors
def repo_get_account(account_ref):
    try:
        return storage_read(account_ref)
    except KeyError as exc:
        raise AccountNotFoundError(f"repository: {account_ref} not found") from exc

# Layer 3: service, translates repository errors to API-facing errors
def service_get_balance(account_ref="ACC-1083"):
    try:
        return repo_get_account(account_ref)
    except AccountNotFoundError as exc:
        raise InvalidTransactionError(f"service: cannot fetch balance for {account_ref}") from exc


try:
    service_get_balance()
except InvalidTransactionError as e:
    chain = []
    cur = e
    while cur is not None:
        chain.append(f"{type(cur).__name__}: {cur}")
        cur = cur.__cause__
    print("Full cause chain (top-level first):")
    for line in chain:
        print(" ->", line)




Full cause chain (top-level first):
 -> InvalidTransactionError: service: cannot fetch balance for ACC-1083
 -> AccountNotFoundError: repository: ACC-1083 not found
 -> KeyError: 'ACC-1083'


### Q84. Implement bounded exponential retry for banking case 2; use sample reference ACC-1084 and explain the result.

In [ ]:
import time

def call_gateway_flaky(account_ref="ACC-1084", _state={"n": 0}):
    _state["n"] += 1
    if _state["n"] < 3:
        raise TransientGatewayError(f"[{account_ref}] transient failure #{_state['n']}")
    return f"settled for {account_ref}"


def retry_with_backoff(fn, max_attempts=4, base_delay=0.01):
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except TransientGatewayError as exc:
            if attempt == max_attempts:
                raise TransientGatewayError(
                    f"giving up after {max_attempts} attempts"
                ) from exc
            delay = base_delay * (2 ** (attempt - 1))
            print(f"  attempt {attempt} failed ({exc}); backing off {delay:.3f}s")
            time.sleep(delay)


print(retry_with_backoff(call_gateway_flaky))




  attempt 1 failed ([ACC-1084] transient failure #1); backing off 0.010s

  attempt 2 failed ([ACC-1084] transient failure #2); backing off 0.020s


settled for ACC-1084


### Q85. Create a transaction context manager for banking case 2; use sample reference ACC-1085 and explain the result.

In [ ]:
class transaction_scope:
    """Commits ledger changes only if the block completes without error."""

    def __init__(self, ledger, account_ref="ACC-1085"):
        self.ledger = ledger
        self.account_ref = account_ref
        self.snapshot = None

    def __enter__(self):
        self.snapshot = dict(self.ledger)
        return self.ledger

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            self.ledger.clear()
            self.ledger.update(self.snapshot)
            print(f"[{self.account_ref}] transaction rolled back due to {exc_type.__name__}: {exc_val}")
            return False  # do not suppress -- caller still sees the exception
        print(f"[{self.account_ref}] transaction committed")
        return False


ledger = {"ACC-1085": 1000}
try:
    with transaction_scope(ledger) as tx:
        tx["ACC-1085"] -= 200
        raise InvalidTransactionError("downstream validation failed")
except InvalidTransactionError as e:
    print(f"REJECTED-> {e}")
print("Ledger after failed transaction:", ledger)




[ACC-1085] transaction rolled back due to InvalidTransactionError: downstream validation failed
REJECTED-> downstream validation failed
Ledger after failed transaction: {'ACC-1085': 1000}


### Q86. Build a dead-letter queue for payment events for banking case 2; use sample reference ACC-1086 and explain the result.

In [ ]:
dead_letter_queue = []

def handle_payment_event(event, account_ref="ACC-1086"):
    try:
        if event["amount"] <= 0:
            raise InvalidTransactionError(f"non-positive amount in event {event}")
        if event.get("gateway_down"):
            raise TransientGatewayError(f"gateway unavailable for event {event}")
        return f"processed {event['amount']} for {account_ref}"
    except InvalidTransactionError as exc:
        # Not retryable -- permanently bad event, park it for manual review.
        dead_letter_queue.append({"event": event, "reason": str(exc), "retryable": False})
        return None
    except TransientGatewayError as exc:
        # Retryable -- park it too, but flagged so a separate worker can retry later.
        dead_letter_queue.append({"event": event, "reason": str(exc), "retryable": True})
        return None


events = [{"amount": 100}, {"amount": -5}, {"amount": 50, "gateway_down": True}]
for e in events:
    print("result:", handle_payment_event(e))

print(f"Dead-letter queue for ACC-1086 has {len(dead_letter_queue)} entries:")
for entry in dead_letter_queue:
    print(" ", entry)




result:

 processed 100 for ACC-1086
result: None
result: None
Dead-letter queue for ACC-1086 has 2 entries:
  {'event': {'amount': -5}, 'reason': "non-positive amount in event {'amount': -5}", 'retryable': False}
  {'event': {'amount': 50, 'gateway_down': True}, 'reason': "gateway unavailable for event {'amount': 50, 'gateway_down': True}", 'retryable': True}


### Q87. Make batch settlement failure-isolated for banking case 2; use sample reference ACC-1087 and explain the result.

In [ ]:
def settle_batch(entries, account_ref="ACC-1087"):
    report = {"settled": [], "failed": []}
    for entry in entries:
        try:
            amount = float(entry["amount"])
            if amount <= 0:
                raise InvalidTransactionError("settlement amount must be positive")
            report["settled"].append({"id": entry["id"], "amount": amount})
        except (KeyError, ValueError, InvalidTransactionError) as exc:
            report["failed"].append({"id": entry.get("id", "<unknown>"), "error": str(exc)})
            continue  # move on to the next entry regardless of this failure
    return report


entries = [
    {"id": "S1", "amount": "100"},
    {"id": "S2", "amount": "-20"},
    {"amount": "50"},          # missing id
    {"id": "S4", "amount": "abc"},
    {"id": "S5", "amount": "75"},
]
report = settle_batch(entries)
print(f"ACC-1087 settlement report: {len(report['settled'])} settled, {len(report['failed'])} failed")
for f in report["failed"]:
    print("  failed:", f)




ACC-1087 settlement report: 2 settled, 3 failed


  failed: {'id': 'S2', 'error': 'settlement amount must be positive'}
  failed: {'id': '<unknown>', 'error': "'id'"}
  failed: {'id': 'S4', 'error': "could not convert string to float: 'abc'"}


### Q88. Report exact paths in nested statements for banking case 2; use sample reference ACC-1088 and explain the result.

In [ ]:
def validate_application(data, account_ref="ACC-1088", path=""):
    """Recursively validate a nested loan-application dict, reporting the exact field path."""
    errors = []
    for key, value in data.items():
        current_path = f"{path}.{key}" if path else key
        if isinstance(value, dict):
            errors.extend(validate_application(value, account_ref, current_path))
        elif value in (None, ""):
            errors.append(f"[{account_ref}] field '{current_path}' is required")
    return errors


application = {
    "applicant": {"name": "A. Rao", "employment": {"employer": "", "years": 3}},
    "loan": {"amount": 200000, "purpose": None},
}

try:
    errors = validate_application(application)
    if errors:
        raise InvalidTransactionError("; ".join(errors))
    print(f"[ACC-1088] application valid")
except InvalidTransactionError as e:
    print(f"REJECTED-> {e}")




REJECTED-> [ACC-1088] field 'applicant.employment.employer' is required; [ACC-1088] field 'loan.purpose' is required


### Q89. Design idempotent recovery after timeout for banking case 2; use sample reference ACC-1089 and explain the result.

In [ ]:
processed_idempotency_keys = {}

def submit_payment(idempotency_key, amount, account_ref="ACC-1089", _simulate_timeout={"n": 0}):
    if idempotency_key in processed_idempotency_keys:
        # We've seen this exact request before -- return the stored result instead of
        # re-charging the account a second time.
        print(f"[{account_ref}] duplicate submission detected, returning cached result")
        return processed_idempotency_keys[idempotency_key]

    _simulate_timeout["n"] += 1
    if _simulate_timeout["n"] == 1:
        # Simulate: the request reached the server and would have succeeded, but the
        # client never got the response (network timeout) and does not know the outcome.
        raise TransientGatewayError(f"[{account_ref}] response timed out, outcome unknown")

    result = f"charged {amount} for {account_ref} (key={idempotency_key})"
    processed_idempotency_keys[idempotency_key] = result
    return result


key = "idem-ACC-1089-001"
try:
    submit_payment(key, 500)
except TransientGatewayError as e:
    print(f"REJECTED-> {e}")
    print("Client retries with the SAME idempotency key:")
    print(submit_payment(key, 500))
print("Client retries again (already processed):")
print(submit_payment(key, 500))




REJECTED-> [ACC-1089] response timed out, outcome unknown
Client retries with the SAME idempotency key:
charged 500 for ACC-1089 (key=idem-ACC-1089-001)
Client retries again (already processed):
[ACC-1089] duplicate submission detected, returning cached result
charged 500 for ACC-1089 (key=idem-ACC-1089-001)


### Q90. Build an exception-safe reconciliation pipeline for banking case 2; use sample reference ACC-1090 and explain the result.

In [ ]:
def fetch_bank_statement(account_ref):
    return [{"id": "T1", "amount": 100}, {"id": "T2", "amount": 250}, {"id": "T3", "amount": -50}]

def fetch_internal_ledger(account_ref):
    return [{"id": "T1", "amount": 100}, {"id": "T2", "amount": 260}]  # T2 mismatched, T3 missing


def reconcile(account_ref="ACC-1090"):
    stage_results = {"fetched_statement": None, "fetched_ledger": None, "diffs": None}
    try:
        stage_results["fetched_statement"] = fetch_bank_statement(account_ref)
        stage_results["fetched_ledger"] = fetch_internal_ledger(account_ref)
    except Exception as exc:
        raise InvalidTransactionError(f"[{account_ref}] reconciliation aborted: could not fetch data") from exc

    diffs = []
    ledger_by_id = {row["id"]: row["amount"] for row in stage_results["fetched_ledger"]}
    for row in stage_results["fetched_statement"]:
        try:
            ledger_amount = ledger_by_id[row["id"]]
            if ledger_amount != row["amount"]:
                diffs.append(f"{row['id']}: statement={row['amount']} ledger={ledger_amount}")
        except KeyError:
            diffs.append(f"{row['id']}: present in statement, missing from ledger")
    stage_results["diffs"] = diffs
    return stage_results


result = reconcile()
print(f"Reconciliation for ACC-1090:")
print(f"  statement rows: {len(result['fetched_statement'])}")
print(f"  ledger rows:    {len(result['fetched_ledger'])}")
print(f"  discrepancies:  {len(result['diffs'])}")
for d in result["diffs"]:
    print("   -", d)



Reconciliation for ACC-1090:


  statement rows: 3
  ledger rows:    2
  discrepancies:  2
   - T2: statement=250 ledger=260
   - T3: present in statement, missing from ledger


### Q91. Design a banking exception hierarchy for banking case 3; use sample reference ACC-1091 and explain the result.

In [ ]:
class AccountError(BankingError):
    """Problems tied to a specific account (not found, frozen, closed)."""

class TransactionError(BankingError):
    """Problems tied to a specific transaction attempt."""

class ComplianceError(BankingError):
    """KYC / regulatory / fraud-monitoring problems."""

class FrozenAccountError(AccountError):
    pass

class DuplicateTransactionError(TransactionError):
    pass


def post_transaction(account_ref, frozen, amount, seen_ids, txn_id):
    if frozen:
        raise FrozenAccountError(f"[{account_ref}] account is frozen")
    if txn_id in seen_ids:
        raise DuplicateTransactionError(f"[{account_ref}] transaction {txn_id} already processed")
    seen_ids.add(txn_id)
    return f"posted {amount} for {account_ref}"


seen = set()
for frozen, amount, txn_id in [(True, 100, "T1"), (False, 100, "T1"), (False, 100, "T2")]:
    try:
        print(post_transaction("ACC-1091", frozen, amount, seen, txn_id))
    except BankingError as e:
        # Catching the common base still tells us the concrete subtype via type(e).
        print(f"REJECTED ({type(e).__name__})-> {e}")




REJECTED (FrozenAccountError)-> [ACC-1091] account is frozen
posted 100 for ACC-1091
posted 100 for ACC-1091


### Q92. Make a transfer rollback-safe for banking case 3; use sample reference ACC-1092 and explain the result.

In [ ]:
def transfer_funds(ledger, from_ref, to_ref, amount):
    original_from = ledger[from_ref]
    original_to = ledger[to_ref]
    try:
        if ledger[from_ref] < amount:
            raise InsufficientFundsError(f"{from_ref} has insufficient funds for {amount}")
        ledger[from_ref] -= amount
        ledger[to_ref] += amount
        if to_ref not in ledger:  # simulated post-debit failure e.g. bad destination
            raise AccountNotFoundError(f"destination {to_ref} vanished mid-transfer")
        return ledger
    except (InsufficientFundsError, AccountNotFoundError):
        # Roll both legs back to their pre-transfer values before re-raising.
        ledger[from_ref] = original_from
        ledger[to_ref] = original_to
        raise


ledger = {"ACC-1092": 1000, "ACC-DEST": 500}
try:
    transfer_funds(ledger, "ACC-1092", "ACC-DEST", 5000)
except InsufficientFundsError as e:
    print(f"REJECTED-> {e}")
print("Ledger unchanged after rollback:", ledger)




REJECTED-> ACC-1092 has insufficient funds for 5000


Ledger unchanged after rollback: {'ACC-1092': 1000, 'ACC-DEST': 500}


### Q93. Preserve causes across service layers for banking case 3; use sample reference ACC-1093 and explain the result.

In [ ]:
# Layer 1: low-level storage
def storage_read(key):
    raise KeyError(key)

# Layer 2: repository, translates storage errors to domain errors
def repo_get_account(account_ref):
    try:
        return storage_read(account_ref)
    except KeyError as exc:
        raise AccountNotFoundError(f"repository: {account_ref} not found") from exc

# Layer 3: service, translates repository errors to API-facing errors
def service_get_balance(account_ref="ACC-1093"):
    try:
        return repo_get_account(account_ref)
    except AccountNotFoundError as exc:
        raise InvalidTransactionError(f"service: cannot fetch balance for {account_ref}") from exc


try:
    service_get_balance()
except InvalidTransactionError as e:
    chain = []
    cur = e
    while cur is not None:
        chain.append(f"{type(cur).__name__}: {cur}")
        cur = cur.__cause__
    print("Full cause chain (top-level first):")
    for line in chain:
        print(" ->", line)

.


Full cause chain (top-level first):
 -> InvalidTransactionError: service: cannot fetch balance for ACC-1093
 -> AccountNotFoundError: repository: ACC-1093 not found
 -> KeyError: 'ACC-1093'


### Q94. Implement bounded exponential retry for banking case 3; use sample reference ACC-1094 and explain the result.

In [ ]:
import time

def call_gateway_flaky(account_ref="ACC-1094", _state={"n": 0}):
    _state["n"] += 1
    if _state["n"] < 3:
        raise TransientGatewayError(f"[{account_ref}] transient failure #{_state['n']}")
    return f"settled for {account_ref}"


def retry_with_backoff(fn, max_attempts=4, base_delay=0.01):
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except TransientGatewayError as exc:
            if attempt == max_attempts:
                raise TransientGatewayError(
                    f"giving up after {max_attempts} attempts"
                ) from exc
            delay = base_delay * (2 ** (attempt - 1))
            print(f"  attempt {attempt} failed ({exc}); backing off {delay:.3f}s")
            time.sleep(delay)


print(retry_with_backoff(call_gateway_flaky))




  attempt 1 failed ([ACC-1094] transient failure #1); backing off 0.010s


  attempt 2 failed ([ACC-1094] transient failure #2); backing off 0.020s

settled for ACC-1094


### Q95. Create a transaction context manager for banking case 3; use sample reference ACC-1095 and explain the result.

In [ ]:
class transaction_scope:
    """Commits ledger changes only if the block completes without error."""

    def __init__(self, ledger, account_ref="ACC-1095"):
        self.ledger = ledger
        self.account_ref = account_ref
        self.snapshot = None

    def __enter__(self):
        self.snapshot = dict(self.ledger)
        return self.ledger

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            self.ledger.clear()
            self.ledger.update(self.snapshot)
            print(f"[{self.account_ref}] transaction rolled back due to {exc_type.__name__}: {exc_val}")
            return False  # do not suppress -- caller still sees the exception
        print(f"[{self.account_ref}] transaction committed")
        return False


ledger = {"ACC-1095": 1000}
try:
    with transaction_scope(ledger) as tx:
        tx["ACC-1095"] -= 200
        raise InvalidTransactionError("downstream validation failed")
except InvalidTransactionError as e:
    print(f"REJECTED-> {e}")
print("Ledger after failed transaction:", ledger)




[ACC-1095] transaction rolled back due to InvalidTransactionError: downstream validation failed


REJECTED-> downstream validation failed
Ledger after failed transaction: {'ACC-1095': 1000}


### Q96. Build a dead-letter queue for payment events for banking case 3; use sample reference ACC-1096 and explain the result.

In [ ]:
dead_letter_queue = []

def handle_payment_event(event, account_ref="ACC-1096"):
    try:
        if event["amount"] <= 0:
            raise InvalidTransactionError(f"non-positive amount in event {event}")
        if event.get("gateway_down"):
            raise TransientGatewayError(f"gateway unavailable for event {event}")
        return f"processed {event['amount']} for {account_ref}"
    except InvalidTransactionError as exc:
        # Not retryable -- permanently bad event, park it for manual review.
        dead_letter_queue.append({"event": event, "reason": str(exc), "retryable": False})
        return None
    except TransientGatewayError as exc:
        # Retryable -- park it too, but flagged so a separate worker can retry later.
        dead_letter_queue.append({"event": event, "reason": str(exc), "retryable": True})
        return None


events = [{"amount": 100}, {"amount": -5}, {"amount": 50, "gateway_down": True}]
for e in events:
    print("result:", handle_payment_event(e))

print(f"Dead-letter queue for ACC-1096 has {len(dead_letter_queue)} entries:")
for entry in dead_letter_queue:
    print(" ", entry)




result: processed 100 for ACC-1096
result: None
result: None
Dead-letter queue for ACC-1096 has 2 entries:
  {'event': {'amount': -5}, 'reason': "non-positive amount in event {'amount': -5}", 'retryable': False}
  {'event': {'amount': 50, 'gateway_down': True}, 'reason': "gateway unavailable for event {'amount': 50, 'gateway_down': True}", 'retryable': True}


### Q97. Make batch settlement failure-isolated for banking case 3; use sample reference ACC-1097 and explain the result.

In [ ]:
def settle_batch(entries, account_ref="ACC-1097"):
    report = {"settled": [], "failed": []}
    for entry in entries:
        try:
            amount = float(entry["amount"])
            if amount <= 0:
                raise InvalidTransactionError("settlement amount must be positive")
            report["settled"].append({"id": entry["id"], "amount": amount})
        except (KeyError, ValueError, InvalidTransactionError) as exc:
            report["failed"].append({"id": entry.get("id", "<unknown>"), "error": str(exc)})
            continue  # move on to the next entry regardless of this failure
    return report


entries = [
    {"id": "S1", "amount": "100"},
    {"id": "S2", "amount": "-20"},
    {"amount": "50"},          # missing id
    {"id": "S4", "amount": "abc"},
    {"id": "S5", "amount": "75"},
]
report = settle_batch(entries)
print(f"ACC-1097 settlement report: {len(report['settled'])} settled, {len(report['failed'])} failed")
for f in report["failed"]:
    print("  failed:", f)




ACC-1097 settlement report: 2 settled, 3 failed
  failed: {'id': 'S2', 'error': 'settlement amount must be positive'}
  failed: {'id': '<unknown>', 'error': "'id'"}
  failed: {'id': 'S4', 'error': "could not convert string to float: 'abc'"}


### Q98. Report exact paths in nested statements for banking case 3; use sample reference ACC-1098 and explain the result.

In [ ]:
def validate_application(data, account_ref="ACC-1098", path=""):
    """Recursively validate a nested loan-application dict, reporting the exact field path."""
    errors = []
    for key, value in data.items():
        current_path = f"{path}.{key}" if path else key
        if isinstance(value, dict):
            errors.extend(validate_application(value, account_ref, current_path))
        elif value in (None, ""):
            errors.append(f"[{account_ref}] field '{current_path}' is required")
    return errors


application = {
    "applicant": {"name": "A. Rao", "employment": {"employer": "", "years": 3}},
    "loan": {"amount": 200000, "purpose": None},
}

try:
    errors = validate_application(application)
    if errors:
        raise InvalidTransactionError("; ".join(errors))
    print(f"[ACC-1098] application valid")
except InvalidTransactionError as e:
    print(f"REJECTED-> {e}")




REJECTED-> [ACC-1098] field 'applicant.employment.employer' is required; [ACC-1098] field 'loan.purpose' is required


### Q99. Design idempotent recovery after timeout for banking case 3; use sample reference ACC-1099 and explain the result.

In [ ]:
processed_idempotency_keys = {}

def submit_payment(idempotency_key, amount, account_ref="ACC-1099", _simulate_timeout={"n": 0}):
    if idempotency_key in processed_idempotency_keys:
        # We've seen this exact request before -- return the stored result instead of
        # re-charging the account a second time.
        print(f"[{account_ref}] duplicate submission detected, returning cached result")
        return processed_idempotency_keys[idempotency_key]

    _simulate_timeout["n"] += 1
    if _simulate_timeout["n"] == 1:
        # Simulate: the request reached the server and would have succeeded, but the
        # client never got the response (network timeout) and does not know the outcome.
        raise TransientGatewayError(f"[{account_ref}] response timed out, outcome unknown")

    result = f"charged {amount} for {account_ref} (key={idempotency_key})"
    processed_idempotency_keys[idempotency_key] = result
    return result


key = "idem-ACC-1099-001"
try:
    submit_payment(key, 500)
except TransientGatewayError as e:
    print(f"REJECTED-> {e}")
    print("Client retries with the SAME idempotency key:")
    print(submit_payment(key, 500))
print("Client retries again (already processed):")
print(submit_payment(key, 500))



REJECTED-> [ACC-1099] response timed out, outcome unknown


Client retries with the SAME idempotency key:
charged 500 for ACC-1099 (key=idem-ACC-1099-001)
Client retries again (already processed):
[ACC-1099] duplicate submission detected, returning cached result
charged 500 for ACC-1099 (key=idem-ACC-1099-001)


### Q100. Build an exception-safe reconciliation pipeline for banking case 3; use sample reference ACC-1100 and explain the result.

In [ ]:
def fetch_bank_statement(account_ref):
    return [{"id": "T1", "amount": 100}, {"id": "T2", "amount": 250}, {"id": "T3", "amount": -50}]

def fetch_internal_ledger(account_ref):
    return [{"id": "T1", "amount": 100}, {"id": "T2", "amount": 260}]  # T2 mismatched, T3 missing


def reconcile(account_ref="ACC-1100"):
    stage_results = {"fetched_statement": None, "fetched_ledger": None, "diffs": None}
    try:
        stage_results["fetched_statement"] = fetch_bank_statement(account_ref)
        stage_results["fetched_ledger"] = fetch_internal_ledger(account_ref)
    except Exception as exc:
        raise InvalidTransactionError(f"[{account_ref}] reconciliation aborted: could not fetch data") from exc

    diffs = []
    ledger_by_id = {row["id"]: row["amount"] for row in stage_results["fetched_ledger"]}
    for row in stage_results["fetched_statement"]:
        try:
            ledger_amount = ledger_by_id[row["id"]]
            if ledger_amount != row["amount"]:
                diffs.append(f"{row['id']}: statement={row['amount']} ledger={ledger_amount}")
        except KeyError:
            diffs.append(f"{row['id']}: present in statement, missing from ledger")
    stage_results["diffs"] = diffs
    return stage_results


result = reconcile()
print(f"Reconciliation for ACC-1100:")
print(f"  statement rows: {len(result['fetched_statement'])}")
print(f"  ledger rows:    {len(result['fetched_ledger'])}")
print(f"  discrepancies:  {len(result['diffs'])}")
for d in result["diffs"]:
    print("   -", d)




Reconciliation for ACC-1100:
  statement rows: 3
  ledger rows:    2
  discrepancies:  2
   - T2: statement=250 ledger=260
   - T3: present in statement, missing from ledger
